<div style="display: flex; align-items: center; justify-content: center; flex-wrap: wrap;">
    <div style="flex: 1; max-width: 400px; display: flex; justify-content: center;">
        <img alt="NOVA IMS logo" src="https://magic.novaims.unl.pt/media/1tdf2arr/ims25_horizontal__positivo_rgb.svg" style="max-width: 70%; height: auto; margin-top: 50px; margin-bottom: 50px;margin-left: 6rem;">
    </div>
    <div style="flex: 2; text-align: center; margin-top: 20px;margin-left: 6rem;">
        <div style="font-size: 28px; font-weight: bold; line-height: 1.2;">
            <span style="color: #FFCD41;">Thesis Project |</span> <span style="color: #F58228;">LISBOA: Lisbon Itinerary System Based On AI</span>
        </div>
        <div style="font-size: 17px; font-weight: bold; margin-top: 10px;">
            2025 / 2026
        </div>
        <div style="font-size: 17px; font-weight: bold;">
            Master in Data Science and Advanced Analytics
        </div>
        <div style="margin-top: 20px;">
            <div>André Filipe Gomes Silvestre, 20240502</div>
            <div style="margin-top: 6px; font-size: 14px;"><b>Supervisors:</b> Prof. Dr. Bruno Jardim; Prof. Dr. Miguel de Castro Neto</div>
        </div>
    </div>
</div>

<style>
@import url('https://fonts.cdnfonts.com/css/avenir-next-lt-pro?styles=29974');
</style>

<div style="background: linear-gradient(to right, #F58228, #FFCD41);
            padding: 15px; color: white; border-radius: 300px; text-align: center;">
    <center><h1 style="margin-left: 100px;margin-top: 10px; margin-bottom: 4px; color: white;
                       font-size: 32px; font-family: 'Avenir Next LT Pro', sans-serif;"><b>API Integration: IPMA, Metro, Carris, CP</b></h1></center>
</div>

<br><br>

## **Notebook overview**

This notebook combines two complementary validation layers for the LISBOA thesis project, updated on **2026-05-18**:

1. raw API and feed exploration for the main weather and mobility sources used by the assistant;
2. a runtime audit of **all 45 LangChain tools** currently exported by `tools.__init__.py`.

The detailed sections focus on IPMA, Metro de Lisboa, Carris Urban, Carris Metropolitana, CP, and the repository's multimodal routing layer. The final audit also covers the non-transport runtime tools from VisitLisboa, Lisboa Aberta, local knowledge search, and constrained web knowledge so the notebook stays aligned with the current LISBOA tool registry.

The goal is to keep one reproducible notebook that is useful for thesis documentation, manual debugging, and quick live checks after the runtime tool layer changes.

<br><br>


In [1]:
# ==========================================================================
# Required libraries (run this cell if not already installed):
# ==========================================================================
# pip install requests pandas

In [2]:
# ==========================================================================
# Master Thesis - API Integration Notebook
#   - André Filipe Gomes Silvestre, 20240502
#
#   This notebook demonstrates API integration and current runtime-tool coverage for:
#     - IPMA (weather forecasts and warnings)
#     - Metro de Lisboa (line status, stations, wait times, frequency)
#     - Carris Metropolitana (suburban bus network and live data)
#     - Carris Urban (GTFS and GTFS-RT for Lisbon buses/trams)
#     - CP Comboios de Portugal (GTFS schedules and live AML status)
#     - LISBOA runtime tools exported by tools.__init__
# ==========================================================================

import json
import time
import warnings
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple

import pandas as pd
import requests
from IPython.display import display

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Set pandas display options for better visualization
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_rows', 50)

# ==========================================================================
# Helper Functions for Visualization
# ==========================================================================

def print_header(title: str, emoji: str = "📊"):
    """Prints a styled header."""
    print(f"\n{'=' * 80}")
    print(f"\033[1m{emoji} {title}\033[0m")
    print(f"{'=' * 80}")


def print_subheader(title: str):
    """Prints a styled subheader."""
    print(f"\n\033[1m{title}\033[0m")
    print("-" * 50)


def print_success(message: str):
    """Prints a success message in green."""
    print(f"\033[1;32m✅ {message}\033[0m")


def print_error(message: str):
    """Prints an error message in red."""
    print(f"\033[1;31m❌ {message}\033[0m")


def print_warning(message: str):
    """Prints a warning message in yellow."""
    print(f"\033[1;33m⚠️ {message}\033[0m")


def print_info(message: str):
    """Prints an info message in blue."""
    print(f"\033[1;34mℹ️ {message}\033[0m")


def format_timestamp(ts: Optional[int]) -> str:
    """Converts Unix timestamp (ms) to readable format."""
    if ts is None:
        return "N/A"
    try:
        return datetime.fromtimestamp(ts / 1000).strftime('%Y-%m-%d %H:%M:%S')
    except Exception:
        return str(ts)


# Request configuration
REQUEST_TIMEOUT = 15  # seconds
MAX_RETRIES = 3
BACKOFF_FACTOR = 2

print_success("Libraries imported and helper functions defined!")
print(f"📅 Notebook execution started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ Libraries imported and helper functions defined!
📅 Notebook execution started: 2026-05-18 13:45:01


## <span style="color: #ffffff;">1. IPMA API (Weather)</span>
<style>
@import url('https://fonts.cdnfonts.com/css/avenir-next-lt-pro?styles=29974');
</style>

<div style="background: transparent;
            padding: 10px; color: #F58228; border-radius: 300px; text-align: center;
            border: 2px solid #F58228;">
    <center><h2 style="margin-left: 120px;margin-top: 10px; margin-bottom: 4px; color: #F58228;
                       font-size: 24px; font-family: 'Avenir Next LT Pro', sans-serif;"><b> 1. IPMA API (Weather)</b></h2></center>
</div>

<br><br>

The IPMA API provides weather forecasts and warnings. For this project, we focus on the Lisbon area.

**Endpoints:**
- **Warnings:** `https://api.ipma.pt/open-data/forecast/warnings/warnings_www.json`
- **Daily Forecast (by location):** `https://api.ipma.pt/open-data/forecast/meteorology/cities/daily/{globalIdLocal}.json`
- **Aggregated Daily Forecast:** `https://api.ipma.pt/open-data/forecast/meteorology/cities/daily/hp-daily-forecast-day{idDay}.json`  <!-- idDay: 0=today, 1=tomorrow, 2=day after tomorrow -->
    - `idDay`: 0=today, 1=tomorrow, 2=day after tomorrow
- **Lisbon Global ID:** `1110600`

### **🔎 Dataset Attributes**

<center><b>Table 1 | </b> IPMA API Attributes. <br>

<br>

|       | **Field**                          | **Description**                                                                 | **Type**     |
|:-----:|:----------------------------------:|:--------------------------------------------------------------------------------|:------------:|
| **1** | `text`                             | Description of the warning (only for yellow/orange/red levels)                  | **`Text`**         |
| **2** | `awarenessTypeName`                | Type of warning (e.g., "Agitação Marítima", "Vento")                            | **`Text`**         |
| **3** | `awarenessLevelID`                 | Warning level color (green, yellow, orange, red)                                | **`Text`**         |
| **4** | `startTime`                        | Start time of the warning                                                       | **`DateTime`**     |
| **5** | `endTime`                          | End time of the warning                                                         | **`DateTime`**     |
| **6** | `precipitaProb`                    | Probability of precipitation (%)                                                | **`Numeric`**      |
| **7** | `tMin`                             | Minimum temperature (°C)                                                        | **`Numeric`**      |
| **8** | `tMax`                             | Maximum temperature (°C)                                                        | **`Numeric`**      |
| **9** | `predWindDir`                      | Predominant wind direction (e.g., N, SW)                                        | **`Text`**         |
| **10**| `forecastDate`                     | Date of the forecast                                                            | **`Date`**         |
</center>

<br><br>

In [3]:
# ==========================================================================
# IPMA API Configuration
# ==========================================================================

# Lisbon Global ID for IPMA API
LISBON_GLOBAL_ID = 1110600

# IPMA Endpoints
IPMA_WARNINGS_URL = "https://api.ipma.pt/open-data/forecast/warnings/warnings_www.json"
IPMA_FORECAST_URL = "https://api.ipma.pt/open-data/forecast/meteorology/cities/daily/{global_id}.json"
IPMA_FORECAST_AGGREGATED_URL = "https://api.ipma.pt/open-data/forecast/meteorology/cities/daily/hp-daily-forecast-day{id_day}.json"  # idDay: 0=today, 1=tomorrow, 2=day after
IPMA_LOCATIONS_URL = "https://api.ipma.pt/open-data/distrits-islands.json"

# Weather type mapping (IPMA codes to descriptions)
WEATHER_TYPES = {
    -99: "No information", 0: "No information", 1: "Clear sky ☀️", 2: "Partly cloudy ⛅",
    3: "Sunny intervals 🌤️", 4: "Cloudy ☁️", 5: "Cloudy (High cloud) ☁️",
    6: "Showers/rain 🌧️", 7: "Light showers 🌦️", 8: "Heavy showers ⛈️",
    9: "Rain 🌧️", 10: "Light rain 🌦️", 11: "Heavy rain 🌧️",
    12: "Intermittent rain 🌧️", 13: "Intermittent light rain 🌦️", 14: "Intermittent heavy rain 🌧️",
    15: "Drizzle 🌧️", 16: "Mist 🌫️", 17: "Fog 🌫️", 18: "Snow ❄️",
    19: "Thunderstorms ⛈️", 20: "Showers and thunderstorms ⛈️", 21: "Hail 🌨️",
    22: "Frost 🥶", 23: "Rain and thunderstorms ⛈️", 24: "Convective clouds ☁️",
    25: "Partly cloudy ⛅", 26: "Fog 🌫️", 27: "Cloudy ☁️",
    28: "Snow showers ❄️", 29: "Rain and snow 🌨️", 30: "Rain and snow 🌨️"
}

# Warning levels with colors and descriptions
WARNING_LEVELS = {
    "green": {"emoji": "🟢", "severity": 0, "description": "No warning", "color": "#28a745"},
    "yellow": {"emoji": "🟡", "severity": 1, "description": "Be aware", "color": "#ffc107"},
    "orange": {"emoji": "🟠", "severity": 2, "description": "Be prepared", "color": "#fd7e14"},
    "red": {"emoji": "🔴", "severity": 3, "description": "Take action", "color": "#dc3545"}
}

# Warning types mapping (Portuguese to English with emojis)
WARNING_TYPES = {
    "Precipitação": {"en": "Precipitation", "emoji": "🌧️"},
    "Trovoada": {"en": "Thunderstorm", "emoji": "⛈️"},
    "Agitação Marítima": {"en": "Rough Sea", "emoji": "🌊"},
    "Vento": {"en": "Wind", "emoji": "💨"},
    "Nevoeiro": {"en": "Fog", "emoji": "🌫️"},
    "Neve": {"en": "Snow", "emoji": "❄️"},
    "Tempo Frio": {"en": "Cold Weather", "emoji": "🥶"},
    "Tempo Quente": {"en": "Hot Weather", "emoji": "🥵"},
    "Ondas": {"en": "Waves", "emoji": "🌊"}
}

# Wind directions
WIND_DIRECTIONS = {
    "N": "North ⬆️", "NE": "Northeast ↗️", "E": "East ➡️", "SE": "Southeast ↘️",
    "S": "South ⬇️", "SW": "Southwest ↙️", "W": "West ⬅️", "NW": "Northwest ↖️",
    "": "Variable 🔄"
}

# Wind Speed Classes
WIND_SPEED_CLASSES = {-99: "N/A", 1: "Weak", 2: "Moderate", 3: "Strong", 4: "Very strong"}

# ==========================================================================
# IPMA API Functions
# ==========================================================================

def get_ipma_warnings(area_filter: str = "LSB") -> Tuple[pd.DataFrame, Dict]:
    """
    Fetches weather warnings from IPMA API.

    Args:
        area_filter (str): Area code to filter (LSB for Lisbon, PTO for Porto, etc.)

    Returns:
        Tuple[pd.DataFrame, Dict]: DataFrame with warnings and metadata dict.
    """
    metadata = {"url": IPMA_WARNINGS_URL, "status": "pending", "total": 0, "filtered": 0}

    try:
        response = requests.get(IPMA_WARNINGS_URL, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        data = response.json()

        metadata["status"] = "success"
        metadata["total"] = len(data)

        df = pd.DataFrame(data)

        if not df.empty:
            # Enrich data
            df['level_emoji'] = df['awarenessLevelID'].map(lambda x: WARNING_LEVELS.get(x, {}).get('emoji', '❓'))
            df['level_desc'] = df['awarenessLevelID'].map(lambda x: WARNING_LEVELS.get(x, {}).get('description', 'Unknown'))
            df['type_emoji'] = df['awarenessTypeName'].map(lambda x: WARNING_TYPES.get(x, {}).get('emoji', '❓'))
            df['type_en'] = df['awarenessTypeName'].map(lambda x: WARNING_TYPES.get(x, {}).get('en', x))

            # Filter for Lisbon area if requested
            if area_filter:
                df_filtered = df[df['idAreaAviso'] == area_filter].copy()
                metadata["filtered"] = len(df_filtered)
                # Also filter out green (no warning) entries for relevance
                df_active = df_filtered[df_filtered['awarenessLevelID'] != 'green']
                return df_active, metadata

        return df, metadata

    except requests.exceptions.RequestException as e:
        metadata["status"] = "error"
        metadata["error"] = str(e)
        print_error(f"Error fetching IPMA warnings: {e}")
        return pd.DataFrame(), metadata


def get_ipma_forecast(global_id: int = LISBON_GLOBAL_ID, days: int = 5) -> Tuple[pd.DataFrame, Dict]:
    """
    Fetches daily weather forecast from IPMA API.

    Args:
        global_id (int): Location global ID (1110600 for Lisbon).
        days (int): Number of days to return (max 5).

    Returns:
        Tuple[pd.DataFrame, Dict]: DataFrame with forecast and metadata dict.
    """
    url = IPMA_FORECAST_URL.format(global_id=global_id)
    metadata = {"url": url, "status": "pending", "location_id": global_id}

    try:
        response = requests.get(url, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        data = response.json()

        metadata["status"] = "success"
        metadata["data_update"] = data.get('dataUpdate', 'N/A')
        metadata["country"] = data.get('country', 'N/A')

        if 'data' in data:
            df = pd.DataFrame(data['data'][:days])

            # Enrich data
            df['weather_desc'] = df['idWeatherType'].map(lambda x: WEATHER_TYPES.get(x, f"Unknown ({x})"))
            df['wind_dir_desc'] = df['predWindDir'].map(lambda x: WIND_DIRECTIONS.get(x, x))
            df['wind_speed_desc'] = df['classWindSpeed'].map(lambda x: WIND_SPEED_CLASSES.get(x, f"Unknown ({x})"))

            # Format precipitation probability
            df['precip_text'] = df['precipitaProb'].apply(lambda x: f"{float(x):.0f}%" if pd.notna(x) else "N/A")

            # Format date nicely
            df['date_formatted'] = pd.to_datetime(df['forecastDate']).dt.strftime('%A, %b %d')

            return df, metadata
        else:
            metadata["status"] = "error"
            metadata["error"] = "No data field in response"
            return pd.DataFrame(), metadata

    except requests.exceptions.RequestException as e:
        metadata["status"] = "error"
        metadata["error"] = str(e)
        print_error(f"Error fetching IPMA forecast: {e}")
        return pd.DataFrame(), metadata


def get_ipma_locations() -> Tuple[pd.DataFrame, Dict]:
    """
    Fetches all available IPMA locations (districts/islands).

    Returns:
        Tuple[pd.DataFrame, Dict]: DataFrame with locations and metadata dict.
    """
    metadata = {"url": IPMA_LOCATIONS_URL, "status": "pending"}

    try:
        response = requests.get(IPMA_LOCATIONS_URL, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        data = response.json()

        metadata["status"] = "success"

        if 'data' in data:
            df = pd.DataFrame(data['data'])
            metadata["total"] = len(df)
            return df, metadata
        else:
            return pd.DataFrame(data), metadata

    except requests.exceptions.RequestException as e:
        metadata["status"] = "error"
        metadata["error"] = str(e)
        print_error(f"Error fetching IPMA locations: {e}")
        return pd.DataFrame(), metadata


def get_ipma_aggregated_forecast(id_day: int = 0) -> Tuple[pd.DataFrame, Dict]:
    """
    Fetches aggregated daily forecast for ALL locations for a specific day.

    This endpoint returns forecast data for all districts/islands at once,
    aggregated by day (today, tomorrow, day after tomorrow).

    Args:
        id_day (int): Day index (0=today, 1=tomorrow, 2=day after tomorrow).

    Returns:
        Tuple[pd.DataFrame, Dict]: DataFrame with forecast for all locations and metadata.
    """
    if id_day not in [0, 1, 2]:
        return pd.DataFrame(), {"status": "error", "error": "id_day must be 0, 1, or 2"}

    url = IPMA_FORECAST_AGGREGATED_URL.format(id_day=id_day)
    day_names = {0: "Today", 1: "Tomorrow", 2: "Day after tomorrow"}
    metadata = {"url": url, "status": "pending", "day": day_names.get(id_day, "Unknown")}

    try:
        response = requests.get(url, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        data = response.json()

        metadata["status"] = "success"
        metadata["data_update"] = data.get('dataUpdate', 'N/A')
        metadata["forecast_date"] = data.get('forecastDate', 'N/A')

        if 'data' in data:
            df = pd.DataFrame(data['data'])
            metadata["total_locations"] = len(df)

            # Enrich data
            df['weather_desc'] = df['idWeatherType'].map(lambda x: WEATHER_TYPES.get(x, f"Unknown ({x})"))
            df['wind_dir_desc'] = df['predWindDir'].map(lambda x: WIND_DIRECTIONS.get(x, x))
            df['precip_text'] = df['precipitaProb'].apply(lambda x: f"{float(x):.0f}%" if pd.notna(x) else "N/A")

            return df, metadata
        else:
            metadata["status"] = "error"
            metadata["error"] = "No data field in response"
            return pd.DataFrame(), metadata

    except requests.exceptions.RequestException as e:
        metadata["status"] = "error"
        metadata["error"] = str(e)
        print_error(f"Error fetching IPMA aggregated forecast: {e}")
        return pd.DataFrame(), metadata


# ==========================================================================
# IPMA API Testing & Visualization
# ==========================================================================

print_header("IPMA API - Weather Data for Portugal", "🌤️")

# Test 1: API Connectivity
print_subheader("1. API Connectivity Test")
start_time = time.time()
response = requests.get(IPMA_FORECAST_URL.format(global_id=LISBON_GLOBAL_ID), timeout=REQUEST_TIMEOUT)
elapsed = time.time() - start_time

if response.status_code == 200:
    print_success(f"Forecast API responding (Status: {response.status_code}, Time: {elapsed:.2f}s)")
else:
    print_error(f"Forecast API error (Status: {response.status_code})")

start_time = time.time()
response = requests.get(IPMA_WARNINGS_URL, timeout=REQUEST_TIMEOUT)
elapsed = time.time() - start_time

if response.status_code == 200:
    print_success(f"Warnings API responding (Status: {response.status_code}, Time: {elapsed:.2f}s)")
else:
    print_error(f"Warnings API error (Status: {response.status_code})")

# Test 2: Weather Forecast for Lisbon
print_subheader("2. Weather Forecast for Lisbon (5-day)")
df_forecast, meta_forecast = get_ipma_forecast(days=5)

if not df_forecast.empty:
    print_info(f"Data last updated: {meta_forecast.get('data_update', 'N/A')}")
    print(f"\n📊 Forecast for Lisbon (Global ID: {LISBON_GLOBAL_ID}):\n")

    # Create a nice display DataFrame
    display_cols = ['date_formatted', 'weather_desc', 'tMin', 'tMax', 'precip_text', 'wind_dir_desc', 'wind_speed_desc']
    df_display = df_forecast[display_cols].copy()
    df_display.columns = ['📅 Date', '🌤️ Weather', '🌡️ Min °C', '🌡️ Max °C', '💧 Precip.', '🧭 Wind Dir', '💨 Wind Speed']
    display(df_display)
else:
    print_error("No forecast data available")

# Test 3: Weather Warnings
print_subheader("3. Active Weather Warnings for Lisbon (LSB)")
df_warnings, meta_warnings = get_ipma_warnings(area_filter="LSB")

print_info(f"Total warnings in API: {meta_warnings.get('total', 0)}")
print_info(f"Filtered for Lisbon (LSB): {meta_warnings.get('filtered', 0)}")

if not df_warnings.empty:
    print_warning(f"⚠️ {len(df_warnings)} active warning(s) for Lisbon:")

    for _, row in df_warnings.iterrows():
        print(f"\n  {row['level_emoji']} {row['type_emoji']} {row['type_en']} - {row['level_desc']}")
        print(f"     Start: {row['startTime']}")
        print(f"     End: {row['endTime']}")
        if row.get('text'):
            print(f"     Details: {row['text']}")
else:
    print_success("No active weather warnings for Lisbon! 🎉")

# Test 4: Available Locations
print_subheader("4. Available IPMA Locations (Districts/Islands)")
df_locations, meta_locations = get_ipma_locations()

if not df_locations.empty:
    print_success(f"Found {len(df_locations)} locations")
    display(df_locations)
else:
    print_error("Could not fetch locations")

# Test 5: Aggregated Daily Forecast (All Locations for a Day)
print_subheader("5. Aggregated Forecast (All Portugal - Today)")
df_aggregated, meta_aggregated = get_ipma_aggregated_forecast(id_day=0)

if not df_aggregated.empty:
    print_success(f"Retrieved forecast for {meta_aggregated.get('total_locations', 0)} locations")
    print_info(f"Forecast date: {meta_aggregated.get('forecast_date', 'N/A')}")
    print_info(f"Data updated: {meta_aggregated.get('data_update', 'N/A')}")

    # Show sample (including Lisbon - globalIdLocal 1110600)
    if 'globalIdLocal' in df_aggregated.columns:
        # Filter for Lisbon area
        lisbon_row = df_aggregated[df_aggregated['globalIdLocal'] == LISBON_GLOBAL_ID]
        if not lisbon_row.empty:
            print(f"\\n📍 Lisbon (ID: {LISBON_GLOBAL_ID}) today:")
            row = lisbon_row.iloc[0]
            print(f"   🌡️ Temperature: {row.get('tMin', 'N/A')}°C - {row.get('tMax', 'N/A')}°C")
            print(f"   🌤️ Weather: {row.get('weather_desc', 'N/A')}")
            print(f"   💧 Precipitation: {row.get('precip_text', 'N/A')}")

    # Show first 5 locations
    print_info("Sample from aggregated forecast (first 5 locations):")
    display_cols = [col for col in ['globalIdLocal', 'weather_desc', 'tMin', 'tMax', 'precip_text'] if col in df_aggregated.columns]
    if display_cols:
        display(df_aggregated[display_cols].head(5))
else:
    print_warning("Could not fetch aggregated forecast")

# Summary
print_subheader("📋 IPMA API Summary")
print("""
┌─────────────────────────────────────────────────────────────────────┐
│ 🌤️  IPMA API Endpoints Tested                                       │
├─────────────────────────────────────────────────────────────────────┤
│ ✅ Daily Forecast:     /forecast/meteorology/cities/daily/<id>.json │
│ ✅ Aggregated Forecast:/forecast/.../hp-daily-forecast-day<d>.json  │
│ ✅ Weather Warnings:   /forecast/warnings/warnings_www.json         │
│ ✅ Locations:          /distrits-islands.json                       │
├─────────────────────────────────────────────────────────────────────┤
│ Key Observations:                                                   │
│ • Forecast data updated hourly                                      │
│ • Up to 5 days of forecast available per location                   │
│ • Aggregated endpoint: all locations for 1 day (idDay: 0,1,2)       │
│ • Warnings cover: Wind, Rain, Thunderstorms, Sea, Fog, Snow, Heat   │
│ • Lisbon Global ID: 1110600                                         │
└─────────────────────────────────────────────────────────────────────┘
""")


🌤️ IPMA API - Weather Data for Portugal

1. API Connectivity Test
--------------------------------------------------
✅ Forecast API responding (Status: 200, Time: 0.04s)
✅ Warnings API responding (Status: 200, Time: 0.02s)

2. Weather Forecast for Lisbon (5-day)
--------------------------------------------------
ℹ️ Data last updated: 2026-05-18T12:31:02

📊 Forecast for Lisbon (Global ID: 1110600):



,📅 Date,🌤️ Weather,🌡️ Min °C,🌡️ Max °C,💧 Precip.,🧭 Wind Dir,💨 Wind Speed
0,"Monday, May 18",Sunny intervals 🌤️,12.4,20.4,0%,Northwest ↖️,Moderate
1,"Tuesday, May 19",Sunny intervals 🌤️,12.8,24.1,4%,Northwest ↖️,Moderate
2,"Wednesday, May 20",Partly cloudy ⛅,14.4,31.6,0%,Northwest ↖️,Moderate
3,"Thursday, May 21",Partly cloudy ⛅,16.7,31.7,0%,Northwest ↖️,Weak
4,"Friday, May 22",Partly cloudy ⛅,16.5,31.3,0%,Northwest ↖️,Moderate



3. Active Weather Warnings for Lisbon (LSB)
--------------------------------------------------
ℹ️ Total warnings in API: 202
ℹ️ Filtered for Lisbon (LSB): 8
✅ No active weather warnings for Lisbon! 🎉

4. Available IPMA Locations (Districts/Islands)
--------------------------------------------------


✅ Found 35 locations


,idRegiao,idAreaAviso,idConcelho,globalIdLocal,latitude,idDistrito,local,longitude
0,1,AVR,5,1010500,40.6413,1,Aveiro,-8.6535
1,1,BJA,5,1020500,38.0200,2,Beja,-7.8700
2,1,BRG,3,1030300,41.5475,3,Braga,-8.4227
3,1,BRG,8,1030800,41.4434,3,Guimarães,-8.2938
4,1,BGC,2,1040200,41.8076,4,Bragança,-6.7606
5,1,CBO,2,1050200,39.8217,5,Castelo Branco,-7.4957
6,1,CBR,3,1060300,40.2081,6,Coimbra,-8.4194
7,1,EVR,5,1070500,38.5701,7,Évora,-7.9104
8,1,FAR,5,1080500,37.0146,8,Faro,-7.9331
9,1,FAR,15,1081505,37.0168,8,Sagres,-8.9403



5. Aggregated Forecast (All Portugal - Today)
--------------------------------------------------


✅ Retrieved forecast for 27 locations
ℹ️ Forecast date: 2026-05-18
ℹ️ Data updated: 2026-05-18T12:31:03
\n📍 Lisbon (ID: 1110600) today:
   🌡️ Temperature: 12°C - 20°C
   🌤️ Weather: Sunny intervals 🌤️
   💧 Precipitation: 0%
ℹ️ Sample from aggregated forecast (first 5 locations):


,globalIdLocal,weather_desc,tMin,tMax,precip_text
0,1010500,Sunny intervals 🌤️,14,19,0%
1,1020500,Cloudy (High cloud) ☁️,9,23,0%
2,1030300,Sunny intervals 🌤️,8,20,2%
3,1040200,Partly cloudy ⛅,7,20,0%
4,1050200,Partly cloudy ⛅,8,22,0%



📋 IPMA API Summary
--------------------------------------------------

┌─────────────────────────────────────────────────────────────────────┐
│ 🌤️  IPMA API Endpoints Tested                                       │
├─────────────────────────────────────────────────────────────────────┤
│ ✅ Daily Forecast:     /forecast/meteorology/cities/daily/<id>.json │
│ ✅ Aggregated Forecast:/forecast/.../hp-daily-forecast-day<d>.json  │
│ ✅ Weather Warnings:   /forecast/warnings/warnings_www.json         │
│ ✅ Locations:          /distrits-islands.json                       │
├─────────────────────────────────────────────────────────────────────┤
│ Key Observations:                                                   │
│ • Forecast data updated hourly                                      │
│ • Up to 5 days of forecast available per location                   │
│ • Aggregated endpoint: all locations for 1 day (idDay: 0,1,2)       │
│ • Warnings cover: Wind, Rain, Thunderstorms, Sea, Fog, Snow, Heat 

### **🧪 IPMA Weather Tools Testing**

This section tests the 4 weather tools currently exported from `tools/ipma_api.py`:

| Tool | Description | Current input surface | Output |
|------|-------------|-----------------------|--------|
| `get_weather_warnings` | Active meteorological warnings | `area` code, default `LSB` for Lisbon | Active warning levels and warning types |
| `get_weather_forecast` | Lisbon daily forecast | `days` and `day_offset` | Temperature, precipitation, wind and forecast date |
| `get_portugal_weather_overview` | Portugal-wide daily overview | `day` (`0`, `1`, or `2`) | Comparable forecast across districts/islands |
| `get_current_weather_summary` | Quick Lisbon summary | none | Today's Lisbon forecast plus warnings |

**Important limitations:**
- The Lisbon daily forecast endpoint currently returns 5 forecast days in the executed API response.
- The aggregated Portugal overview endpoint is used for `day` values `0`, `1`, and `2`.
- Weather warnings are provider-bounded and should not be extrapolated beyond the IPMA response.
- Refresh cadence should be read from the returned `dataUpdate` timestamp; the notebook should not assume a fixed hourly update schedule.

In [4]:
# ==========================================================================
# IPMA - Weather Tools Testing (4 tools)
# ==========================================================================

import os
import sys

# Add tools directory to path
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../../tools')))

# Import IPMA tools
try:
    from ipma_api import (
        get_current_weather_summary,
        get_portugal_weather_overview,
        get_weather_forecast,
        get_weather_warnings,
    )
    IPMA_TOOLS_AVAILABLE = True
except ImportError as e:
    print_error(f"Could not import IPMA tools: {e}")
    IPMA_TOOLS_AVAILABLE = False

if IPMA_TOOLS_AVAILABLE:
    print_header("IPMA - WEATHER TOOLS TESTING", "🌤️")

    # =======================================================================
    # Test 1: get_weather_warnings - Active warnings for Lisbon
    # =======================================================================
    print_subheader("1. get_weather_warnings - Weather Warnings")

    print("\n⚠️ Test 1.1: Warnings for Lisbon (LSB)")
    result = get_weather_warnings.invoke({"area": "LSB"})
    print(result)

    print("\n⚠️ Test 1.2: Warnings for Porto (PTO) - comparison")
    result = get_weather_warnings.invoke({"area": "PTO"})
    print(result)

    # =======================================================================
    # Test 2: get_weather_forecast - Daily forecast for Lisbon
    # =======================================================================
    print_subheader("2. get_weather_forecast - Weather Forecast")

    print("\n📅 Test 2.1: 3-day forecast (default)")
    result = get_weather_forecast.invoke({"days": 3})
    print(result)

    print("\n📅 Test 2.2: 5-day forecast (API maximum)")
    result = get_weather_forecast.invoke({"days": 5})
    print(result)

    print("\n📅 Test 2.3: 1-day forecast (today only)")
    result = get_weather_forecast.invoke({"days": 1})
    print(result)

    # =======================================================================
    # Test 3: get_portugal_weather_overview - Portugal-wide overview
    # =======================================================================
    print_subheader("3. get_portugal_weather_overview - Portugal Overview")

    print("\n🇵🇹 Test 3.1: Weather today across Portugal (day=0)")
    result = get_portugal_weather_overview.invoke({"day": 0})
    print(result)

    print("\n🇵🇹 Test 3.2: Weather tomorrow across Portugal (day=1)")
    result = get_portugal_weather_overview.invoke({"day": 1})
    print(result)

    print("\n🇵🇹 Test 3.3: Weather day after tomorrow (day=2)")
    result = get_portugal_weather_overview.invoke({"day": 2})
    print(result)

    # =======================================================================
    # Test 4: get_current_weather_summary - Quick summary for Lisbon
    # =======================================================================
    print_subheader("4. get_current_weather_summary - Quick Summary")

    print("\n📊 Test 4.1: Combined summary (forecast + warnings)")
    result = get_current_weather_summary.invoke({})
    print(result)

    # =======================================================================
    # Summary
    # =======================================================================
    print("\n" + "=" * 75)
    print_success("All 4 IPMA weather tools tested successfully!")
    print("\n📊 Summary:")
    print("   ✅ get_weather_warnings - Weather warnings working")
    print("   ✅ get_weather_forecast - Daily forecast (1-5 days) working")
    print("   ✅ get_portugal_weather_overview - National overview working")
    print("   ✅ get_current_weather_summary - Quick summary working")
    print("\n📡 IPMA API endpoints used:")
    print("   • https://api.ipma.pt/open-data/forecast/warnings/warnings_www.json")
    print("   • https://api.ipma.pt/open-data/forecast/meteorology/cities/daily/1110600.json")
    print("   • https://api.ipma.pt/open-data/forecast/meteorology/cities/daily/hp-daily-forecast-day{0,1,2}.json")
    print("\n💡 Note: Forecasts limited to 5 days by IPMA API.")

else:
    print_error("IPMA tools not available - skipping tests")

❌ Could not import IPMA tools: No module named 'ipma_api'
❌ IPMA tools not available - skipping tests


## <span style="color: #ffffff;">2. Metro de Lisboa API</span>
<style>
@import url('https://fonts.cdnfonts.com/css/avenir-next-lt-pro?styles=29974');
</style>

<div style="background: transparent;
            padding: 10px; color: white; border-radius: 300px; text-align: center;
            border: 2px solid #F58228;">
    <center><h2 style="margin-left: 120px;margin-top: 10px; margin-bottom: 4px; color: #F58228;
                       font-size: 24px; font-family: 'Avenir Next LT Pro', sans-serif;"><b> 2. Metro de Lisboa API</b></h2></center>
</div>

<br><br>

Metro de Lisboa provides an **official OAuth2-authenticated API** for line status, station information, wait times, destinations, and interval metadata. The LISBOA runtime also keeps no-auth fallbacks for public line status and station coordinates so the assistant can still answer non-live questions when official credentials are not configured.

**Official API Portal:** `https://api.metrolisboa.pt/store/`  
**Official API Base:** `https://api.metrolisboa.pt:8243/estadoServicoML/1.0.1/`

### **📡 Official API Endpoints (OAuth2 Required)**

| Endpoint | Method | Description | Auth |
|----------|--------|-------------|------|
| `/tempoEspera/Estacao/todos` | GET | Wait times for ALL stations at once | OAuth2 |
| `/tempoEspera/Estacao/{estacao}` | GET | Wait times at a specific station | OAuth2 |
| `/tempoEspera/Linha/{linha}` | GET | Wait times for entire line | OAuth2 |
| `/infoEstacao/todos` | GET | All stations with GPS coordinates | OAuth2 |
| `/infoEstacao/{estacao}` | GET | Specific station info | OAuth2 |
| `/infoDestinos/todos` | GET | All destination names | OAuth2 |
| `/estadoLinha/todos` | GET | All line operational status | OAuth2 |
| `/estadoLinha/{linha}` | GET | Specific line status | OAuth2 |
| `/infoIntervalos/{linha}/{dia}` | GET | Train intervals by line/day type | OAuth2 |
| `/infoIntervalos/{linha}/{dia}/{hora}` | GET | Train intervals by line/day/hour | OAuth2 |

> **Note:** `{linha}` = Amarela, Azul, Verde, Vermelha | `{dia}` = S (weekday), F (weekend/holiday) | `{hora}` = hhmmss format

### **📡 Fallback Endpoints (No Auth)**

| Endpoint | Description |
|----------|-------------|
| `app.metrolisboa.pt/status/getLinhas.php` | Line status (unofficial) |
| `ArcGIS POITransportes/FeatureServer` | Station GeoJSON with coordinates |

> Runtime note: official realtime calls require `METRO_CONSUMER_KEY` and `METRO_CONSUMER_SECRET`. When credentials or live service evidence are unavailable, runtime tools should label the result as limited rather than inventing waits.

### **🔎 Dataset Attributes**

<center><b>Table 2 | </b> Metro de Lisboa Waiting Times API Attributes. <br>

<br>

|       | **Field**                          | **Description**                                                                 | **Type**     |
|:-----:|:----------------------------------:|:--------------------------------------------------------------------------------|:-----------:|
| **1** | `stop_id`                          | Station code (e.g., CG = Campo Grande)                                          | **`Text`**         |
| **2** | `destino`                          | Destination ID (maps to station name)                                           | **`Integer`**      |
| **3** | `tempoChegada1`                    | Waiting time for next train (seconds)                                           | **`Integer`**      |
| **4** | `tempoChegada2`                    | Waiting time for second train (seconds)                                         | **`Integer`**      |
| **5** | `tempoChegada3`                    | Waiting time for third train (seconds)                                          | **`Integer`**      |
| **6** | `stop_name`                        | Station name                                                                     | **`Text`**         |
| **7** | `stop_lat` / `stop_lon`            | GPS coordinates                                                                  | **`Float`**        |
| **8** | `linha`                            | Lines served (e.g., "[Amarela, Verde]")                                          | **`Text`**         |

</center>

<br><br>

In [5]:
# ==========================================================================
# Metro de Lisboa API Configuration
# ==========================================================================

# --- Official Metro de Lisboa API (OAuth2 authenticated) ---
# Register at: https://api.metrolisboa.pt/store/
# Documentation: https://api.metrolisboa.pt/store/apis/info?name=EstadoServicoML&version=1.0.1
METRO_API_BASE = "https://api.metrolisboa.pt:8243/estadoServicoML/1.0.1"
METRO_TOKEN_URL = "https://api.metrolisboa.pt:8243/token"

# OAuth2 credentials (load from environment or config)
# Register at https://api.metrolisboa.pt/store/ to get your credentials
import os

METRO_CONSUMER_KEY = os.getenv("METRO_CONSUMER_KEY", "")
METRO_CONSUMER_SECRET = os.getenv("METRO_CONSUMER_SECRET", "")

# Official API Endpoints (OAuth2 required):
# - /estadoLinha/todos          - All line statuses
# - /estadoLinha/{linha}        - Specific line status (Amarela, Azul, Verde, Vermelha)
# - /tempoEspera/Estacao/{id}   - Waiting times at station (e.g., CG = Campo Grande)
# - /tempoEspera/Linha/{linha}  - Waiting times for entire line
# - /infoEstacao/todos          - All station info with GPS coordinates
# - /infoDestinos/todos         - All destination names

# Station ID mapping (name -> API code)
METRO_STATION_IDS = {
    "rato": "RA", "marquês de pombal": "MP", "picoas": "PI", "saldanha": "SA",
    "campo pequeno": "CP", "entre campos": "EC", "cidade universitária": "CU",
    "campo grande": "CG", "quinta das conchas": "QC", "lumiar": "LU",
    "ameixoeira": "AX", "senhor roubado": "SR", "odivelas": "OD",
    "santa apolónia": "SP", "terreiro do paço": "TP", "baixa-chiado": "BC",
    "restauradores": "RE", "avenida": "AV", "parque": "PA", "são sebastião": "SS",
    "praça de espanha": "PE", "jardim zoológico": "JZ", "laranjeiras": "LA",
    "alto dos moinhos": "AH", "colégio militar": "CM", "carnide": "CA",
    "pontinha": "PO", "alfornelos": "AF", "amadora este": "AS", "reboleira": "RB",
    "cais do sodré": "CS", "rossio": "RO", "martim moniz": "MM", "intendente": "IN",
    "anjos": "AN", "arroios": "AR", "alameda": "AM", "areeiro": "AE", "roma": "RM",
    "alvalade": "AL", "telheiras": "TE", "olaias": "OL", "bela vista": "BV",
    "chelas": "CH", "olivais": "OS", "cabo ruivo": "CR", "oriente": "OR",
    "moscavide": "MO", "encarnação": "EN", "aeroporto": "AP"
}

# Destination ID mapping (from /infoDestinos/todos)
METRO_DESTINATIONS = {
    "33": "Reboleira", "34": "Amadora Este", "35": "Pontinha", "36": "Colégio Militar/Luz",
    "37": "Laranjeiras", "38": "São Sebastião", "39": "Avenida", "40": "Baixa-Chiado",
    "41": "Terreiro do Paço", "42": "Santa Apolónia", "43": "Odivelas", "44": "Lumiar",
    "45": "Campo Grande", "46": "Campo Pequeno", "48": "Rato", "50": "Telheiras",
    "51": "Alvalade", "52": "Alameda", "53": "Martim Moniz", "54": "Cais do Sodré",
    "56": "Bela Vista", "57": "Chelas", "59": "Moscavide", "60": "Aeroporto"
}

# --- Fallback APIs (no auth required) ---
# Metro Status API (Unofficial but reliable fallback)
METRO_STATUS_URL = "https://app.metrolisboa.pt/status/getLinhas.php"

# Metro Stations GeoJSON (Official data from Lisboa municipal ArcGIS)
METRO_STATIONS_GEOJSON_URL = "https://services.arcgis.com/1dSrzEWVQn5kHHyK/arcgis/rest/services/POITransportes/FeatureServer/1/query?outFields=*&where=1%3D1&f=geojson"

# Line key mapping (API uses Portuguese names)
LINE_KEY_MAP = {
    "Amarela": "amarela",
    "Azul": "azul",
    "Verde": "verde",
    "Vermelha": "vermelha"
}

# Status translations
STATUS_TRANSLATIONS = {
    "Ok": {"emoji": "✅", "en": "Operating normally"},
    "Serviço Condicionado": {"emoji": "⚠️", "en": "Reduced service"},
    "Serviço Encerrado": {"emoji": "❌", "en": "Service closed"},
    "Interrompido": {"emoji": "🚫", "en": "Service interrupted"},
    " Ok": {"emoji": "✅", "en": "Operating normally"},  # Note the space before Ok
}

# ==========================================================================
# Metro de Lisboa API Functions
# ==========================================================================

def haversine_distance(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """
    Calculate the great-circle distance between two points on Earth using the Haversine formula.

    Args:
        lat1, lon1: Latitude and longitude of point 1 (degrees).
        lat2, lon2: Latitude and longitude of point 2 (degrees).

    Returns:
        float: Distance in kilometers.
    """
    import math
    R = 6371  # Earth's radius in kilometers

    lat1_rad = math.radians(lat1)
    lat2_rad = math.radians(lat2)
    delta_lat = math.radians(lat2 - lat1)
    delta_lon = math.radians(lon2 - lon1)

    a = math.sin(delta_lat/2)**2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(delta_lon/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

    return R * c


def calculate_line_distance(df_stations: pd.DataFrame, line_name: str) -> float:
    """
    Calculate the approximate total distance of a Metro line from station coordinates.

    Uses the sum of straight-line distances between consecutive stations.
    Note: This is an approximation as actual track may curve.

    Args:
        df_stations: DataFrame with station data including latitude and longitude.
        line_name: Name of the line (e.g., 'Amarela', 'Azul').

    Returns:
        float: Approximate distance in kilometers.
    """
    # Filter stations for this line (handle interchange stations with '/')
    line_stations = df_stations[
        df_stations['line'].str.contains(line_name, case=False, na=False)
    ].copy()

    if len(line_stations) < 2:
        return 0.0

    # Sort stations by latitude for a rough geographic ordering
    # (This is an approximation - ideally we'd have station sequence data)
    line_stations = line_stations.sort_values('latitude', ascending=False)

    total_distance = 0.0
    prev_row = None

    for _, row in line_stations.iterrows():
        if prev_row is not None and row['latitude'] is not None and row['longitude'] is not None:
            dist = haversine_distance(
                prev_row['latitude'], prev_row['longitude'],
                row['latitude'], row['longitude']
            )
            total_distance += dist
        prev_row = row

    return round(total_distance, 1)


def get_metro_stations_geojson() -> Tuple[Dict, pd.DataFrame, Dict]:
    """
    Fetches Metro station data from Lisboa municipal GeoJSON API.

    Returns:
        Tuple[Dict, pd.DataFrame, Dict]: GeoJSON data, DataFrame of stations, and metadata.
    """
    metadata = {"url": METRO_STATIONS_GEOJSON_URL, "status": "pending"}

    try:
        response = requests.get(METRO_STATIONS_GEOJSON_URL, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        data = response.json()

        metadata["status"] = "success"

        if 'features' in data:
            stations = []
            for feature in data['features']:
                props = feature.get('properties', {})
                geom = feature.get('geometry', {})
                coords = geom.get('coordinates', [None, None])

                stations.append({
                    'name': props.get('NOME', 'Unknown'),
                    'line': props.get('LINHA', 'Unknown'),
                    'status': props.get('SITUACAO', 'Unknown'),
                    'code': props.get('COD_SIG', 'Unknown'),
                    'longitude': coords[0] if len(coords) > 0 else None,
                    'latitude': coords[1] if len(coords) > 1 else None,
                    'is_interchange': '/' in str(props.get('LINHA', ''))
                })

            df = pd.DataFrame(stations)
            metadata["total_stations"] = len(stations)
            return data, df, metadata
        else:
            metadata["error"] = "No features in GeoJSON"
            return {}, pd.DataFrame(), metadata

    except requests.exceptions.RequestException as e:
        metadata["status"] = "error"
        metadata["error"] = str(e)
        print_error(f"Error fetching Metro stations GeoJSON: {e}")
        return {}, pd.DataFrame(), metadata


def get_metro_status(df_stations: pd.DataFrame = None) -> Tuple[pd.DataFrame, Dict]:
    """
    Fetches real-time Metro de Lisboa line status and combines with station data.

    Args:
        df_stations: Optional DataFrame with station data for statistics.

    Returns:
        Tuple[pd.DataFrame, Dict]: DataFrame with line status and metadata.
    """
    metadata = {"url": METRO_STATUS_URL, "status": "pending", "timestamp": datetime.now().isoformat()}

    try:
        response = requests.get(METRO_STATUS_URL, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        data = response.json()

        metadata["status"] = "success"

        if 'resposta' in data:
            status_data = data['resposta']
            formatted_data = []

            # Line metadata (routes, colors) - distances calculated from coordinates
            line_metadata = {
                "amarela": {"name": "Yellow Line", "emoji": "🟡", "color": "#FFCD41", "route": "Rato ↔ Odivelas"},
                "azul": {"name": "Blue Line", "emoji": "🔵", "color": "#0075BF", "route": "Santa Apolónia ↔ Reboleira"},
                "verde": {"name": "Green Line", "emoji": "🟢", "color": "#00A651", "route": "Telheiras ↔ Cais do Sodré"},
                "vermelha": {"name": "Red Line", "emoji": "🔴", "color": "#ED1C24", "route": "S. Sebastião ↔ Aeroporto"}
            }

            for line_key, line_info in line_metadata.items():
                raw_status = status_data.get(line_key, 'Unknown')
                status_info = STATUS_TRANSLATIONS.get(raw_status, {"emoji": "❓", "en": raw_status})

                # Count stations and calculate distance from real data if available
                stations_count = 0
                distance_km = 0.0
                if df_stations is not None and not df_stations.empty:
                    # Count stations belonging to this line (including shared ones)
                    line_pt = line_key.title()  # Convert to Portuguese format
                    stations_count = len(df_stations[df_stations['line'].str.contains(line_pt, case=False, na=False)])
                    # Calculate distance from coordinates using Haversine formula
                    distance_km = calculate_line_distance(df_stations, line_pt)

                formatted_data.append({
                    'line': line_key.title(),
                    'line_emoji': line_info['emoji'],
                    'line_name': line_info['name'],
                    'route': line_info['route'],
                    'stations': stations_count,
                    'distance_km': distance_km,
                    'status_raw': raw_status.strip(),
                    'status_emoji': status_info['emoji'],
                    'status': status_info['en']
                })

            return pd.DataFrame(formatted_data), metadata
        else:
            metadata["status"] = "error"
            metadata["error"] = "Unexpected response format"
            return pd.DataFrame(), metadata

    except requests.exceptions.RequestException as e:
        metadata["status"] = "error"
        metadata["error"] = str(e)
        print_error(f"Error fetching Metro status: {e}")
        return pd.DataFrame(), metadata


def analyze_metro_network(df_stations: pd.DataFrame) -> Dict:
    """
    Analyzes Metro network statistics from station data.

    Args:
        df_stations: DataFrame with station data.

    Returns:
        Dict: Network statistics.
    """
    if df_stations.empty:
        return {}

    stats = {
        "total_stations": len(df_stations),
        "interchange_stations": len(df_stations[df_stations['is_interchange']]),
        "unique_stations": len(df_stations[~df_stations['is_interchange']]) + len(df_stations[df_stations['is_interchange']]),
        "lines": {},
        "interchange_list": []
    }

    # Get interchange stations
    interchanges = df_stations[df_stations['is_interchange']]
    stats["interchange_list"] = interchanges[['name', 'line']].to_dict('records')

    # Count stations per line
    for line in ['Amarela', 'Azul', 'Verde', 'Vermelha']:
        count = len(df_stations[df_stations['line'].str.contains(line, case=False, na=False)])
        stats["lines"][line] = count

    return stats


# ==========================================================================
# Metro de Lisboa Official API - OAuth2 Authentication Functions
# ==========================================================================

# Token cache (in-memory)
_metro_access_token = None
_metro_token_expiry = None

def get_metro_access_token(force_refresh: bool = False) -> Optional[str]:
    """
    Gets a valid OAuth2 access token for the Metro de Lisboa Official API.

    Implements OAuth2 Client Credentials flow. Tokens are valid for 3600 seconds.

    Args:
        force_refresh: Force token refresh even if current token is valid.

    Returns:
        Optional[str]: Access token if successful, None otherwise.
    """
    global _metro_access_token, _metro_token_expiry

    if not METRO_CONSUMER_KEY or not METRO_CONSUMER_SECRET:
        return None

    # Check if current token is still valid (with 5-minute buffer)
    if not force_refresh and _metro_access_token and _metro_token_expiry:
        if datetime.now() < _metro_token_expiry - timedelta(minutes=5):
            return _metro_access_token

    try:
        import base64

        import urllib3
        urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

        credentials = f"{METRO_CONSUMER_KEY}:{METRO_CONSUMER_SECRET}"
        encoded_credentials = base64.b64encode(credentials.encode()).decode()

        headers = {
            "Authorization": f"Basic {encoded_credentials}",
            "Content-Type": "application/x-www-form-urlencoded"
        }

        response = requests.post(
            METRO_TOKEN_URL,
            headers=headers,
            data={"grant_type": "client_credentials"},
            timeout=REQUEST_TIMEOUT,
            verify=False
        )

        if response.status_code != 200:
            return None

        token_data = response.json()
        _metro_access_token = token_data.get("access_token")
        expires_in = token_data.get("expires_in", 3600)
        _metro_token_expiry = datetime.now() + timedelta(seconds=expires_in)

        return _metro_access_token

    except Exception:
        return None


def metro_api_request(endpoint: str, params: Dict = None) -> Optional[Dict]:
    """
    Makes an authenticated request to the Metro de Lisboa Official API.

    Args:
        endpoint: API endpoint (e.g., '/tempoEspera/Estacao/CG').
        params: Optional query parameters.

    Returns:
        Optional[Dict]: JSON response if successful, None otherwise.
    """
    token = get_metro_access_token()
    if not token:
        return None

    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

    url = f"{METRO_API_BASE}{endpoint}"
    headers = {"accept": "application/json", "Authorization": f"Bearer {token}"}

    try:
        response = requests.get(url, headers=headers, params=params, timeout=REQUEST_TIMEOUT, verify=False)

        if response.status_code == 401:
            token = get_metro_access_token(force_refresh=True)
            if token:
                headers["Authorization"] = f"Bearer {token}"
                response = requests.get(url, headers=headers, params=params, timeout=REQUEST_TIMEOUT, verify=False)

        if response.status_code == 200:
            return response.json()
        return None

    except Exception:
        return None


def format_wait_time(seconds: int) -> str:
    """Formats waiting time in seconds to human-readable string."""
    if seconds <= 0:
        return "arriving"
    elif seconds < 60:
        return f"{seconds}s"
    elif seconds < 120:
        return f"1 min {seconds - 60}s"
    else:
        minutes = seconds // 60
        remaining = seconds % 60
        return f"{minutes} min {remaining}s" if remaining > 0 else f"{minutes} min"


def get_metro_wait_times(station_id: str) -> Tuple[List[Dict], Dict]:
    """
    Gets real-time waiting times for a specific Metro station.

    Args:
        station_id: Station code (e.g., 'CG' for Campo Grande).

    Returns:
        Tuple[List[Dict], Dict]: List of waiting times and metadata.
    """
    metadata = {"endpoint": f"/tempoEspera/Estacao/{station_id}", "status": "pending"}

    data = metro_api_request(f"/tempoEspera/Estacao/{station_id}")

    if not data or data.get("codigo") != "200":
        metadata["status"] = "error"
        metadata["error"] = "API request failed or credentials not configured"
        return [], metadata

    metadata["status"] = "success"
    wait_data = data.get("resposta", [])

    formatted = []
    for entry in wait_data:
        dest_id = entry.get("destino", "")
        dest_name = METRO_DESTINATIONS.get(dest_id, f"Dest {dest_id}")

        try:
            wait1 = int(entry.get("tempoChegada1", "0"))
            wait2 = int(entry.get("tempoChegada2", "0"))
            wait3 = int(entry.get("tempoChegada3", "0"))
        except (ValueError, TypeError):
            continue

        if wait1 > 0:
            formatted.append({
                "destination": dest_name,
                "next_train": format_wait_time(wait1),
                "following": [format_wait_time(wait2), format_wait_time(wait3)],
                "seconds": [wait1, wait2, wait3]
            })

    return formatted, metadata


# ==========================================================================
# Metro de Lisboa API Testing & Visualization
# ==========================================================================

print_header("Metro de Lisboa API - Real-time Status", "🚇")

# Test 1: API Connectivity
print_subheader("1. API Connectivity Test")
start_time = time.time()
try:
    response = requests.get(METRO_STATUS_URL, timeout=REQUEST_TIMEOUT)
    elapsed = time.time() - start_time
    if response.status_code == 200:
        print_success(f"Metro Status API responding (Status: {response.status_code}, Time: {elapsed:.2f}s)")
    else:
        print_error(f"Metro Status API error (Status: {response.status_code})")
except Exception as e:
    print_error(f"Metro Status API connection failed: {e}")

# Test 2: Station Data (GeoJSON) - Fetch first to use in status
print_subheader("2. Metro Stations (GeoJSON Data)")
geojson_data, df_metro_stations, meta_stations = get_metro_stations_geojson()

if not df_metro_stations.empty:
    print_success(f"Found {len(df_metro_stations)} Metro stations from GeoJSON API")

    # Analyze network
    network_stats = analyze_metro_network(df_metro_stations)

    print("\n📊 Station Distribution by Line:")
    for line, count in network_stats.get("lines", {}).items():
        emoji = {"Amarela": "🟡", "Azul": "🔵", "Verde": "🟢", "Vermelha": "🔴"}.get(line, "⚪")
        print(f"   {emoji} {line}: {count} stations")

    print(f"\n🔄 Interchange Stations ({network_stats.get('interchange_stations', 0)}):")
    for interchange in network_stats.get("interchange_list", []):
        print(f"   • {interchange['name']} ({interchange['line']})")

    # Display sample stations
    print("\n\033[1mSample Metro Stations:\033[0m")
    display(df_metro_stations[['name', 'line', 'status', 'latitude', 'longitude']].head(10))
else:
    print_warning("Could not fetch Metro stations data")
    network_stats = {}

# Test 3: Line Status (use station data for counts)
print_subheader("3. Current Line Status")
df_metro, meta_metro = get_metro_status(df_metro_stations if not df_metro_stations.empty else None)

if not df_metro.empty:
    print_info(f"Data retrieved at: {meta_metro.get('timestamp', 'N/A')}")
    print()

    # Create visual status display
    for _, row in df_metro.iterrows():
        status_color = "\033[1;32m" if row['status_emoji'] == '✅' else "\033[1;33m" if row['status_emoji'] == '⚠️' else "\033[1;31m"
        print(f"  {row['line_emoji']} {row['line_name']:<12} │ {status_color}{row['status_emoji']} {row['status']}\033[0m")
        print(f"     └── Route: {row['route']} ({row['stations']} stations, ~{row['distance_km']:.1f}km calculated)")

    # Display as table
    print("\n\033[1mDetailed Status Table:\033[0m")
    print("\033[3m(Distances calculated from station coordinates using Haversine formula)\033[0m")
    display_cols = ['line_emoji', 'line_name', 'route', 'stations', 'distance_km', 'status_emoji', 'status']
    df_display = df_metro[display_cols].copy()
    df_display.columns = ['🚇', 'Line', 'Route', 'Stations', 'Dist (km)', '📊', 'Status']
    display(df_display)
else:
    print_error("Could not fetch Metro status")

# Test 4: Network Statistics (from real data)
print_subheader("4. Metro Network Statistics")

# Calculate from real data
total_lines = len(df_metro) if not df_metro.empty else 0
total_stations = network_stats.get('total_stations', 0) if network_stats else 0
interchange_count = network_stats.get('interchange_stations', 0) if network_stats else 0
total_distance = df_metro['distance_km'].sum() if not df_metro.empty else 0

# Helper for aligned rows
def metro_row(text):
    return f"│ {text:<67}│"

print("┌─────────────────────────────────────────────────────────────────────┐")
print(metro_row("🚇 Metro de Lisboa Network Overview (from API data)"))
print("├─────────────────────────────────────────────────────────────────────┤")
print(metro_row(f"Total Lines:              {total_lines:>4}"))
print(metro_row(f"Total Stations:           {total_stations:>4} (from GeoJSON API)"))
print(metro_row(f"Interchange Stations:     {interchange_count:>4} (shared between lines)"))
print(metro_row(f"Total Network Distance:   {total_distance:>4.1f} km (from coordinates)"))
print("├─────────────────────────────────────────────────────────────────────┤")

# Display line details from real data
if not df_metro.empty:
    for _, row in df_metro.iterrows():
        line_text = f"{row['line_emoji']} {row['line_name']}: {row['route']}"
        stats_text = f"({row['stations']} stations, ~{row['distance_km']:.1f}km)"
        combined = f"{line_text:<45} {stats_text}"
        print(metro_row(combined))

print("├─────────────────────────────────────────────────────────────────────┤")
print(metro_row("Operating Hours: ~06:30 - 01:00 (varies by station)"))
print(metro_row("Frequency: 4-8 minutes (peak), 8-12 minutes (off-peak)"))
print("└─────────────────────────────────────────────────────────────────────┘")

# Test 5: Official API with Waiting Times (OAuth2)
print_subheader("5. Official API - Waiting Times (OAuth2)")

# Check if credentials are configured
has_credentials = bool(METRO_CONSUMER_KEY and METRO_CONSUMER_SECRET)

if has_credentials:
    print_success("OAuth2 credentials configured")

    # Test token acquisition
    token = get_metro_access_token()
    if token:
        print_success(f"Access token acquired (expires: {_metro_token_expiry.strftime('%H:%M:%S') if _metro_token_expiry else 'N/A'})")

        # Test waiting times for a sample station (Campo Grande)
        test_station = "CG"  # Campo Grande (interchange station)
        wait_times, meta_wait = get_metro_wait_times(test_station)

        if wait_times:
            print_success(f"Waiting times retrieved for Campo Grande ({len(wait_times)} directions)")
            print("\n⏱️ Real-time Waiting Times at Campo Grande:")
            for wt in wait_times:
                emoji = "🟡" if wt["destination"] in ["Odivelas", "Rato"] else "🟢"
                print(f"   {emoji} → {wt['destination']}: {wt['next_train']} (then {wt['following'][0]}, {wt['following'][1]})")
        else:
            print_warning(f"No waiting times available: {meta_wait.get('error', 'Unknown')}")
    else:
        print_error("Failed to acquire access token")
else:
    print_warning("OAuth2 credentials not configured")
    print("\n   To enable waiting times API:")
    print("   1. Register at: https://api.metrolisboa.pt/store/")
    print("   2. Create an application and get credentials")
    print("   3. Set environment variables:")
    print("      METRO_CONSUMER_KEY=your_key")
    print("      METRO_CONSUMER_SECRET=your_secret")
    print("\n   Available Official API endpoints:")
    print("   • /tempoEspera/Estacao/{id} - Wait times per station")
    print("   • /tempoEspera/Linha/{linha} - Wait times per line")
    print("   • /infoEstacao/todos - All stations with GPS")
    print("   • /estadoLinha/todos - Line status (also available via fallback)")

# Summary
print_subheader("📋 Metro de Lisboa API Summary")

# Helper for summary rows
def summary_row(text):
    return f"│ {text:<67}│"

print("┌─────────────────────────────────────────────────────────────────────┐")
print(summary_row("🚇 Metro de Lisboa API Endpoints Tested"))
print("├─────────────────────────────────────────────────────────────────────┤")
print(summary_row("Fallback APIs (no auth required):"))
print(summary_row("✅ Status API:        app.metrolisboa.pt/status/getLinhas.php"))
print(summary_row("✅ Stations GeoJSON:  ArcGIS POITransportes/FeatureServer"))
print("├─────────────────────────────────────────────────────────────────────┤")
print(summary_row("Official API (OAuth2): api.metrolisboa.pt:8243"))
api_status = "✅" if has_credentials else "⚠️"
print(summary_row(f"{api_status} /tempoEspera/Estacao/{{id}} - Real-time waiting times"))
print(summary_row(f"{api_status} /tempoEspera/Linha/{{linha}} - Line waiting times"))
print(summary_row(f"{api_status} /infoEstacao/todos - Station info with GPS"))
print(summary_row("   Register at: api.metrolisboa.pt/store/"))
print("├─────────────────────────────────────────────────────────────────────┤")
print(summary_row("Data Retrieved:"))
print(summary_row(f"   • Total Stations: {total_stations} (real-time from API)"))
print(summary_row(f"   • Lines Monitored: {total_lines}"))
print(summary_row(f"   • Interchange Stations: {interchange_count}"))
print("├─────────────────────────────────────────────────────────────────────┤")
print(summary_row("Key Observations:"))
print(summary_row("• Official API available with free OAuth2 registration"))
print(summary_row("• Fallback APIs work without authentication"))
print(summary_row("• GeoJSON provides station names, lines, and coordinates"))
print(summary_row("• Distances calculated via Haversine formula (approx.)"))
print("└─────────────────────────────────────────────────────────────────────┘")


🚇 Metro de Lisboa API - Real-time Status

1. API Connectivity Test
--------------------------------------------------


✅ Metro Status API responding (Status: 200, Time: 0.11s)



2. Metro Stations (GeoJSON Data)
--------------------------------------------------


✅ Found 50 Metro stations from GeoJSON API

📊 Station Distribution by Line:
   🟡 Amarela: 13 stations
   🔵 Azul: 18 stations
   🟢 Verde: 13 stations
   🔴 Vermelha: 12 stations

🔄 Interchange Stations (6):
   • Baixa Chiado (Azul/Verde)
   • Marquês de Pombal (Azul/Amarela)
   • São Sebastião (Azul/Vermelha)
   • Saldanha (Amarela/Vermelha)
   • Alameda (Verde/Vermelha)
   • Campo Grande (Amarela/Verde)

Sample Metro Stations:


,name,line,status,latitude,longitude
0,Cais do Sodré,Verde,Linha existente,38.706115,-9.146208
1,Terreiro do Paço,Azul,Linha existente,38.707129,-9.134202
2,Baixa Chiado,Azul/Verde,Linha existente,38.710570,-9.140151
3,Santa Apolónia,Azul,Linha existente,38.713924,-9.122334
4,Rossio,Verde,Linha existente,38.714039,-9.137886
5,Restauradores,Azul,Linha existente,38.715922,-9.142118
6,Martim Moniz,Verde,Linha existente,38.717702,-9.135738
7,Avenida,Azul,Linha existente,38.719403,-9.145152
8,Rato,Amarela,Linha existente,38.720158,-9.154688
9,Intendente,Verde,Linha existente,38.723298,-9.135185



3. Current Line Status
--------------------------------------------------
ℹ️ Data retrieved at: 2026-05-18T13:45:02.677278

  🟡 Yellow Line  │ ✅ Operating normally
     └── Route: Rato ↔ Odivelas (13 stations, ~9.7km calculated)
  🔵 Blue Line    │ ✅ Operating normally
     └── Route: Santa Apolónia ↔ Reboleira (18 stations, ~22.6km calculated)
  🟢 Green Line   │ ✅ Operating normally
     └── Route: Telheiras ↔ Cais do Sodré (13 stations, ~8.7km calculated)
  🔴 Red Line     │ ✅ Operating normally
     └── Route: S. Sebastião ↔ Aeroporto (12 stations, ~12.5km calculated)

Detailed Status Table:
(Distances calculated from station coordinates using Haversine formula)


,🚇,Line,Route,Stations,Dist (km),📊,Status
0,🟡,Yellow Line,Rato ↔ Odivelas,13,9.7,✅,Operating normally
1,🔵,Blue Line,Santa Apolónia ↔ Reboleira,18,22.6,✅,Operating normally
2,🟢,Green Line,Telheiras ↔ Cais do Sodré,13,8.7,✅,Operating normally
3,🔴,Red Line,S. Sebastião ↔ Aeroporto,12,12.5,✅,Operating normally



4. Metro Network Statistics
--------------------------------------------------
┌─────────────────────────────────────────────────────────────────────┐
│ 🚇 Metro de Lisboa Network Overview (from API data)                 │
├─────────────────────────────────────────────────────────────────────┤
│ Total Lines:                 4                                     │
│ Total Stations:             50 (from GeoJSON API)                  │
│ Interchange Stations:        6 (shared between lines)              │
│ Total Network Distance:   53.5 km (from coordinates)               │
├─────────────────────────────────────────────────────────────────────┤
│ 🟡 Yellow Line: Rato ↔ Odivelas                (13 stations, ~9.7km)│
│ 🔵 Blue Line: Santa Apolónia ↔ Reboleira       (18 stations, ~22.6km)│
│ 🟢 Green Line: Telheiras ↔ Cais do Sodré       (13 stations, ~8.7km)│
│ 🔴 Red Line: S. Sebastião ↔ Aeroporto          (12 stations, ~12.5km)│
├──────────────────────────────────────────────────────────────

## <span style="color: #ffffff;">3. Carris Metropolitana API</span>
<style>
@import url('https://fonts.cdnfonts.com/css/avenir-next-lt-pro?styles=29974');
</style>

<div style="background: transparent;
            padding: 10px; color: white; border-radius: 300px; text-align: center;
            border: 2px solid #F58228;">
    <center><h2 style="margin-left: 120px;margin-top: 10px; margin-bottom: 4px; color: #F58228;
                       font-size: 24px; font-family: 'Avenir Next LT Pro', sans-serif;"><b> 3. Carris Metropolitana API</b></h2></center>
</div>

<br><br>

Carris Metropolitana provides open JSON and GTFS/GTFS-RT-backed endpoints for the metropolitan bus network. The current repository runtime uses the unversioned base URL `https://api.carrismetropolitana.pt`; the official repository documents the versioned form `https://api.carrismetropolitana.pt/v1/[endpoint]`. In this execution, `/lines` returned the same item count through both forms, and `/gtfs` redirected successfully to the current GTFS zip endpoint.

**API repository/documentation:** [carrismetropolitana/api](https://github.com/carrismetropolitana/api)  
**Runtime base URL:** `https://api.carrismetropolitana.pt`

> **Coverage note:** do not describe AML as having 23 municipalities. The current `/municipalities` endpoint returned 23 municipality records, while the official API documentation describes the service as covering bus transit data for 15 of the 18 municipalities of the Lisbon metropolitan area. Treat the endpoint records as API coverage metadata, not as the administrative count of AML municipalities.

> **Implementation note:** collection endpoints such as `/alerts`, `/vehicles`, `/stops`, `/lines`, `/routes`, `/municipalities`, and `/datasets/facilities/*` are tested directly. Detail-only endpoints such as `/patterns/:id`, `/shapes/:id`, and `/stops/:id/realtime` require a real identifier and are tested with representative IDs rather than as collection URLs.

### **📡 Available Endpoints**

| Endpoint | Tested as | Description |
|----------|-----------|-------------|
| `/gtfs` | collection/file | Current GTFS feed zip |
| `/alerts` | collection | Service alerts and disruptions |
| `/vehicles` | collection | Vehicle records; active vehicles include realtime position and trip fields |
| `/stops` | collection | Stops with coordinates, municipality, route, line and accessibility metadata |
| `/stops/:id/realtime` | detail example | Realtime/scheduled arrivals at a stop |
| `/lines` | collection | Bus line metadata |
| `/routes` | collection | Route information |
| `/patterns/:id` | detail example | Detailed journey pattern |
| `/shapes/:id` | detail example | Route geometry as GeoJSON |
| `/municipalities` | collection | Municipality records exposed by the API |
| `/datasets/facilities/encm` | collection | Customer-service locations and live waiting metadata |
| `/datasets/facilities/schools` | collection | School locations |

### **🔎 Dataset Attributes**

<center><b>Table 3.1 | </b> Carris Metropolitana <strong>Alerts</strong> Attributes. <br>

<br>

|       | **Field**                          | **Description**                                                                 | **Type**     |
|:-----:|:----------------------------------:|:--------------------------------------------------------------------------------|:-----------:|
| **1** | `alert_id`                         | Unique identifier for the alert                                                 | **`Text`**     |
| **2** | `cause`                            | Cause of the alert                                                              | **`Text`**     |
| **3** | `effect`                           | Effect of the alert                                                             | **`Text`**     |
| **4** | `description_text`                 | Detailed translated description of the alert                                    | **`Object`**   |
| **5** | `header_text`                      | Short translated summary of the alert                                           | **`Object`**   |
| **6** | `informed_entity`                  | Affected trips, routes, stops, lines or agencies                                | **`List`**     |
| **7** | `active_period`                    | Time periods when the alert is active                                           | **`List`**     |

</center>

<br>

<center><b>Table 3.2 | </b> Carris Metropolitana <strong>Vehicles</strong> Attributes. <br>

<br>

|       | **Field**                          | **Description**                                                                 | **Type**     |
|:-----:|:----------------------------------:|:--------------------------------------------------------------------------------|:-----------:|
| **1** | `id` / `agency_id`                 | Vehicle and operator identifiers                                                | **`Text`**     |
| **2** | `lat` / `lon`                      | Current GPS coordinates when the vehicle is active                              | **`Numeric`**  |
| **3** | `speed` / `bearing`                | Current speed and heading                                                       | **`Numeric`**  |
| **4** | `timestamp`                        | Last known position timestamp                                                   | **`Numeric`**  |
| **5** | `line_id` / `route_id` / `pattern_id` | Current service identifiers                                                  | **`Text`**     |
| **6** | `trip_id` / `stop_id`              | Current trip and stop context                                                   | **`Text`**     |
| **7** | `current_status` / `schedule_relationship` | GTFS-RT operating status                                              | **`Text`**     |
| **8** | `shift_id` / `block_id`            | Operational block identifiers                                                   | **`Text`**     |
| **9** | `make` / `model` / `license_plate` | Vehicle fleet metadata                                                          | **`Text`**     |
| **10**| `capacity_*`, `wheelchair_accessible`, `bikes_allowed`, `contactless` | Capacity and accessibility metadata                         | **`Mixed`**    |

</center>

<br>

<center><b>Table 3.3 | </b> Carris Metropolitana <strong>Stops</strong> Attributes. <br>

<br>

|       | **Field**                          | **Description**                                                                 | **Type**     |
|:-----:|:----------------------------------:|:--------------------------------------------------------------------------------|:-----------:|
| **1** | `id` / `stop_id`                   | Stop identifiers                                                                | **`Text`**     |
| **2** | `name` / `short_name` / `tts_name` | Stop names for display and speech                                               | **`Text`**     |
| **3** | `lat` / `lon`                      | GPS coordinates (WGS84)                                                         | **`Numeric`**  |
| **4** | `lines` / `routes` / `patterns`    | Lines, routes and journey patterns serving the stop                             | **`List`**     |
| **5** | `locality`, `parish_*`, `municipality_*`, `district_*`, `region_*` | Administrative geography metadata                         | **`Mixed`**    |
| **6** | `facilities`                       | Linked facilities such as schools or customer-service locations                 | **`List`**     |
| **7** | `operational_status`               | Current operational status of the stop                                          | **`Text`**     |
| **8** | `wheelchair_boarding`              | Accessibility flag                                                              | **`Text`**     |

</center>

<br><br>

In [6]:
# ==========================================================================
# Carris Metropolitana API Configuration
# ==========================================================================

# Base URL used by the current runtime tools.
# The public API also exposes versioned aliases, but the repository tools use this base.
CARRIS_BASE_URL = "https://api.carrismetropolitana.pt"
CARRIS_BASE_URL_V1 = CARRIS_BASE_URL
CARRIS_BASE_URL_V2 = f"{CARRIS_BASE_URL}/v2"

# Collection endpoints that should answer directly with HTTP 200.
# Detail-only endpoints such as /patterns/:id and /shapes/:id are tested separately.
CARRIS_ENDPOINTS = {
    "gtfs": {"url": f"{CARRIS_BASE_URL}/gtfs", "desc": "GTFS feed zip download"},
    "alerts": {"url": f"{CARRIS_BASE_URL}/alerts", "desc": "Service alerts and disruptions"},
    "vehicles": {"url": f"{CARRIS_BASE_URL}/vehicles", "desc": "Real-time vehicle positions"},
    "stops": {"url": f"{CARRIS_BASE_URL}/stops", "desc": "Stop information (~12,000+ stops)"},
    "lines": {"url": f"{CARRIS_BASE_URL}/lines", "desc": "Line information (~700+ lines)"},
    "routes": {"url": f"{CARRIS_BASE_URL}/routes", "desc": "Route information"},
    "patterns": {"url": f"{CARRIS_BASE_URL}/patterns", "desc": "Pattern collection endpoint (detail lookup uses /patterns/:id)"},
    "municipalities": {"url": f"{CARRIS_BASE_URL}/municipalities", "desc": "Municipality information"},
    "encm": {"url": f"{CARRIS_BASE_URL}/datasets/facilities/encm", "desc": "Customer service locations"},
    "schools": {"url": f"{CARRIS_BASE_URL}/datasets/facilities/schools", "desc": "School locations"},
}

CARRIS_DETAIL_ENDPOINT_NOTES = pd.DataFrame(
    [
        {"endpoint": "/stops/:id/realtime", "status": "ID-required", "tested_by": "get_carris_stop_realtime"},
        {"endpoint": "/patterns/:id", "status": "ID-required", "tested_by": "get_carris_pattern"},
        {"endpoint": "/shapes/:id", "status": "ID-required", "tested_by": "get_carris_shape"},
    ]
)

# Alert effect translations
ALERT_EFFECTS = {
    "NO_SERVICE": {"emoji": "🚫", "desc": "No service"},
    "REDUCED_SERVICE": {"emoji": "⚠️", "desc": "Reduced service"},
    "SIGNIFICANT_DELAYS": {"emoji": "⏰", "desc": "Significant delays"},
    "DETOUR": {"emoji": "↩️", "desc": "Route detour"},
    "ADDITIONAL_SERVICE": {"emoji": "➕", "desc": "Additional service"},
    "MODIFIED_SERVICE": {"emoji": "🔄", "desc": "Modified service"},
    "OTHER_EFFECT": {"emoji": "ℹ️", "desc": "Other effect"},
    "UNKNOWN_EFFECT": {"emoji": "❓", "desc": "Unknown effect"},
    "STOP_MOVED": {"emoji": "📍", "desc": "Stop moved"},
}

# Alert cause translations
ALERT_CAUSES = {
    "DEMONSTRATION": "Demonstration/Protest",
    "ACCIDENT": "Accident",
    "CONSTRUCTION": "Construction works",
    "WEATHER": "Weather conditions",
    "SPECIAL_EVENT": "Special event",
    "HOLIDAY": "Holiday",
    "MAINTENANCE": "Maintenance",
    "POLICE_ACTIVITY": "Police activity",
    "TECHNICAL_PROBLEM": "Technical problem",
    "UNKNOWN_CAUSE": "Unknown cause",
    "OTHER_CAUSE": "Other cause",
}

# ==========================================================================
# Carris Metropolitana API Functions
# ==========================================================================

def get_carris_data(endpoint: str, max_items: Optional[int] = None) -> Tuple[pd.DataFrame, Dict]:
    """
    Generic function to fetch data from Carris Metropolitana API.

    Args:
        endpoint (str): API endpoint name (e.g., 'alerts', 'stops', 'lines').
        max_items (int, optional): Maximum number of items to return.

    Returns:
        Tuple[pd.DataFrame, Dict]: DataFrame with data and metadata dict.
    """
    endpoint_info = CARRIS_ENDPOINTS.get(endpoint)
    if not endpoint_info:
        return pd.DataFrame(), {"status": "error", "error": f"Unknown endpoint: {endpoint}"}

    url = endpoint_info["url"]
    metadata = {"url": url, "endpoint": endpoint, "status": "pending"}

    for attempt in range(MAX_RETRIES):
        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()
            data = response.json()

            metadata["status"] = "success"
            metadata["total"] = len(data)

            df = pd.DataFrame(data)

            if max_items and not df.empty:
                df = df.head(max_items)
                metadata["returned"] = len(df)

            return df, metadata

        except requests.exceptions.RequestException as e:
            if attempt < MAX_RETRIES - 1:
                time.sleep(BACKOFF_FACTOR ** attempt)
            else:
                metadata["status"] = "error"
                metadata["error"] = str(e)
                print_error(f"Error fetching {endpoint}: {e}")
                return pd.DataFrame(), metadata

    return pd.DataFrame(), metadata


def get_carris_stop_realtime(stop_id: str) -> Tuple[pd.DataFrame, Dict]:
    """
    Fetches real-time arrivals for a specific stop.

    Args:
        stop_id (str): Stop ID (e.g., '060001').

    Returns:
        Tuple[pd.DataFrame, Dict]: DataFrame with arrivals and metadata.
    """
    url = f"{CARRIS_BASE_URL}/stops/{stop_id}/realtime"
    metadata = {"url": url, "stop_id": stop_id, "status": "pending"}

    try:
        response = requests.get(url, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        data = response.json()

        metadata["status"] = "success"
        metadata["arrivals"] = len(data)

        return pd.DataFrame(data), metadata

    except requests.exceptions.RequestException as e:
        metadata["status"] = "error"
        metadata["error"] = str(e)
        return pd.DataFrame(), metadata


def get_carris_pattern(pattern_id: str) -> Tuple[Dict, Dict]:
    """
    Fetches detailed pattern information.

    Args:
        pattern_id (str): Pattern ID.

    Returns:
        Tuple[Dict, Dict]: Pattern data and metadata.
    """
    url = f"{CARRIS_BASE_URL}/patterns/{pattern_id}"
    metadata = {"url": url, "pattern_id": pattern_id, "status": "pending"}

    try:
        response = requests.get(url, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        data = response.json()

        metadata["status"] = "success"
        return data, metadata

    except requests.exceptions.RequestException as e:
        metadata["status"] = "error"
        metadata["error"] = str(e)
        return {}, metadata


def get_carris_shape(shape_id: str) -> Tuple[Dict, Dict]:
    """
    Fetches shape geometry for a route.

    Args:
        shape_id (str): Shape ID.

    Returns:
        Tuple[Dict, Dict]: Shape data and metadata.
    """
    url = f"{CARRIS_BASE_URL}/shapes/{shape_id}"
    metadata = {"url": url, "shape_id": shape_id, "status": "pending"}

    try:
        response = requests.get(url, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        data = response.json()

        metadata["status"] = "success"
        return data, metadata

    except requests.exceptions.RequestException as e:
        metadata["status"] = "error"
        metadata["error"] = str(e)
        return {}, metadata


# ==========================================================================
# Carris Metropolitana API Testing & Visualization
# ==========================================================================

print_header("Carris Metropolitana API - Bus Network Data", "🚌")

# Test 1: API Connectivity for all endpoints
print_subheader("1. API Connectivity Test (Collection Endpoints)")

endpoint_status = []
for name, info in CARRIS_ENDPOINTS.items():
    start_time = time.time()
    try:
        response = requests.get(info["url"], timeout=10)
        elapsed = time.time() - start_time
        status = "✅" if response.status_code == 200 else "❌"
        endpoint_status.append({
            "Endpoint": name,
            "Status": status,
            "Code": response.status_code,
            "Time (s)": f"{elapsed:.2f}",
            "Description": info["desc"]
        })
    except Exception:
        endpoint_status.append({
            "Endpoint": name,
            "Status": "❌",
            "Code": "Error",
            "Time (s)": "N/A",
            "Description": info["desc"]
        })

df_endpoints = pd.DataFrame(endpoint_status)
display(df_endpoints)

print_info("Detail-only endpoints are not collection URLs; they require concrete IDs and are tested below.")
display(CARRIS_DETAIL_ENDPOINT_NOTES)

# Test 2: Alerts
print_subheader("2. Service Alerts")
df_alerts, meta_alerts = get_carris_data("alerts")

if not df_alerts.empty:
    print_success(f"Found {meta_alerts.get('total', 0)} alerts")

    # Process alerts for display
    if 'effect' in df_alerts.columns:
        df_alerts['effect_emoji'] = df_alerts['effect'].map(lambda x: ALERT_EFFECTS.get(x, {}).get('emoji', '❓'))
        df_alerts['effect_desc'] = df_alerts['effect'].map(lambda x: ALERT_EFFECTS.get(x, {}).get('desc', x))

    if 'cause' in df_alerts.columns:
        df_alerts['cause_desc'] = df_alerts['cause'].map(lambda x: ALERT_CAUSES.get(x, x))

    # Show first 5 alerts
    display_cols = [col for col in ['alert_id', 'effect_emoji', 'effect_desc', 'cause_desc', 'header_text'] if col in df_alerts.columns]
    if display_cols:
        display(df_alerts[display_cols].head(5))
    else:
        display(df_alerts.head(5))
else:
    print_info("No active alerts")

# Test 3: Municipalities
print_subheader("3. Municipalities Served")
df_municipalities, meta_mun = get_carris_data("municipalities")

if not df_municipalities.empty:
    print_success(f"Serving {len(df_municipalities)} municipalities")
    display(df_municipalities)
else:
    print_error("Could not fetch municipalities")

# Test 4: Stops Summary
print_subheader("4. Bus Stops Network")
df_stops, meta_stops = get_carris_data("stops", max_items=10)

if meta_stops.get("total"):
    print_success(f"Total bus stops: {meta_stops.get('total', 0):,}")
    print_info("Showing first 10 stops:")

    display_cols = [col for col in ['id', 'name', 'locality', 'municipality_name', 'lat', 'lon'] if col in df_stops.columns]
    if display_cols:
        display(df_stops[display_cols])
    else:
        display(df_stops.head(10))
else:
    print_error("Could not fetch stops")

# Test 5: Lines Summary
print_subheader("5. Bus Lines Network")
df_lines, meta_lines = get_carris_data("lines", max_items=10)

if meta_lines.get("total"):
    print_success(f"Total bus lines: {meta_lines.get('total', 0):,}")
    print_info("Showing first 10 lines:")

    display_cols = [col for col in ['id', 'short_name', 'long_name', 'color'] if col in df_lines.columns]
    if display_cols:
        display(df_lines[display_cols])
    else:
        display(df_lines.head(10))
else:
    print_error("Could not fetch lines")

# Test 6: Routes Summary
print_subheader("6. Bus Routes")
df_routes, meta_routes = get_carris_data("routes", max_items=10)

if meta_routes.get("total"):
    print_success(f"Total routes: {meta_routes.get('total', 0):,}")
    print_info("Showing first 10 routes:")
    display(df_routes.head(10))
else:
    print_error("Could not fetch routes")

# Test 7: Customer Service (ENCM)
print_subheader("7. Customer Service Locations (ENCM)")
df_encm, meta_encm = get_carris_data("encm")

if not df_encm.empty:
    print_success(f"Found {len(df_encm)} customer service locations")
    display(df_encm)
else:
    print_info("No ENCM data available")



🚌 Carris Metropolitana API - Bus Network Data

1. API Connectivity Test (Collection Endpoints)
--------------------------------------------------


,Endpoint,Status,Code,Time (s),Description
0,gtfs,✅,200,5.56,GTFS feed zip download
1,alerts,✅,200,0.09,Service alerts and disruptions
2,vehicles,✅,200,0.13,Real-time vehicle positions
3,stops,✅,200,0.11,"Stop information (~12,000+ stops)"
4,lines,✅,200,0.08,Line information (~700+ lines)
5,routes,✅,200,0.09,Route information
6,patterns,✅,200,0.07,Pattern collection endpoint (detail lookup uses /patterns/:id)
7,municipalities,✅,200,0.05,Municipality information
8,encm,✅,200,0.09,Customer service locations
9,schools,✅,200,0.08,School locations


ℹ️ Detail-only endpoints are not collection URLs; they require concrete IDs and are tested below.


,endpoint,status,tested_by
0,/stops/:id/realtime,ID-required,get_carris_stop_realtime
1,/patterns/:id,ID-required,get_carris_pattern
2,/shapes/:id,ID-required,get_carris_shape



2. Service Alerts
--------------------------------------------------
✅ Found 69 alerts


...


6. Bus Routes
--------------------------------------------------
✅ Total routes: 946
ℹ️ Showing first 10 routes:


,color,facilities,id,line_id,localities,long_name,municipalities,patterns,short_name,text_color
0,#C61D23,[],1001_0,1001,"[Alfragide, Amadora, Reboleira, Buraca]",Alfragide (Estr Seminario) - Reboleira (Estação),[1115],"[1001_0_1, 1001_0_2]",1001,#FFFFFF
1,#C61D23,[],1002_0,1002,"[Reboleira, Amadora, Atalaia, Alfragide]",Reboleira (Estação) | Circular via Alfragide (Centro comercial ) e Amadora (Estação Sul),[1115],[1002_0_3],1002,#FFFFFF
2,#C61D23,[],1003_0,1003,"[Amadora, Amadora Este]",Amadora (Estação Norte) - Amadora Este (Metro),[1115],"[1003_0_1, 1003_0_2]",1003,#FFFFFF
3,#C61D23,[],1004_0,1004,"[Amadora, Moinhos da Funcheira]",Amadora (Estação Norte) via Moinhos da Funcheira | Circular,[1115],[1004_0_3],1004,#FFFFFF
4,#C61D23,[],1005_0,1005,"[Amadora, Casal da Mira]",Amadora (Estação Norte) - UBBO,[1115],"[1005_0_1, 1005_0_2]",1005,#FFFFFF
5,#C61D23,[],1005_1,1005,[Amadora],Casal São Brás - Amadora (Estação Norte),[1115],[1005_1_2],1005,#FFFFFF
6,#C61D23,[],1005_2,1005,[Amadora],Amadora (Estação Norte) | Circular madrugada,[1115],[1005_2_3],1005,#FFFFFF
7,#C61D23,[],1006_0,1006,"[Amadora, Moinhos da Funcheira, Casal da Mira]",Amadora (Estação Norte) - UBBO | Noturna,[1115],"[1006_0_1, 1006_0_2]",1006,#FFFFFF
8,#C61D23,[],1006_1,1006,"[Amadora, Moinhos da Funcheira]",Amadora (Estação Norte) - Moinhos Funcheira (Rotunda) | Noturna,[1115],"[1006_1_1, 1006_1_2]",1006,#FFFFFF
9,#3D85C6,[],1008_0,1008,"[Amadora Este, Amadora, None, Venda Nova]",Amadora Este (Metro) | Circular,[1115],[1008_0_3],1008,#FFFFFF



7. Customer Service Locations (ENCM)
--------------------------------------------------


✅ Found 26 customer service locations


,active_counters,address,brand_name,current_ratio,current_status,currently_waiting,district_id,district_name,email,expected_wait_time,google_place_id,hours_friday,hours_monday,hours_saturday,hours_special,hours_sunday,hours_thursday,hours_tuesday,hours_wednesday,id,is_open,lat,locality,lon,municipality_id,municipality_name,name,parish_id,parish_name,phone,postal_code,region_id,region_name,short_name,stops,url
0,1,Avenida José Elias Garcia 71,Espaço navegante®,1.000000,open,0,11,Lisboa,,0,ChIJf55o2a3NHg0RNBNFOsMt3gw,[08:00-19:00],[08:00-19:00],[],,[],[08:00-19:00],[08:00-19:00],[08:00-19:00],8400000000000001,True,38.756310,Queluz,-9.253493,1111,Sintra,Espaço navegante® Queluz,26,União das freguesias de Queluz e Belas,210410400,2745-155,PT170,AML,Queluz,"[170927, 170928, 172063, 172476, 172495, 172499]",https://www.carrismetropolitana.pt/encm
1,2,Avenida Santos Mattos 8A,Espaço navegante®,0.090909,busy,22,11,Lisboa,,2200,ChIJEc3j1VPNHg0Rk3GL2GCM9MQ,[08:00-19:00],[08:00-19:00],[],,[],[08:00-19:00],[08:00-19:00],[08:00-19:00],8400000000000002,True,38.758672,Amadora,-9.235156,1115,Amadora,Espaço navegante® Amadora (Estação),17,Venteira,210410400,2700-336,PT170,AML,Amadora (Estação),"[030317, 030319, 030469, 030475, 030477, 030685, 030821, 030845, 030869, 030880, 030687, 030843,...",https://www.carrismetropolitana.pt/encm
2,2,"Largo Alfredo Dinis, Cacilhas",Espaço navegante®,2.000000,open,0,15,Setúbal,,0,ChIJiU442pA1GQ0Ra-zWOkQd4uM,[08:00-19:00],[08:00-19:00],[],,[],[08:00-19:00],[08:00-19:00],[08:00-19:00],8400000000000003,True,38.687925,Almada,-9.147518,1503,Almada,Espaço navegante® Cacilhas,12,"União das freguesias de Almada, Cova da Piedade, Pragal e Cacilhas",210410400,2800-252,PT170,AML,Cacilhas,"[020037, 020113, 020219, 020363, 020403, 020429, 020449, 020489, 020955, 020959, 020973, 021007,...",https://www.carrismetropolitana.pt/encm
3,2,Praça do Brasil,Espaço navegante®,1.000000,open,2,15,Setúbal,,200,ChIJ3wbD7bBDGQ0R6U4uPeHaG3I,[08:00-21:00],[08:00-21:00],[08:00-21:00],,[08:00-21:00],[08:00-21:00],[08:00-21:00],[08:00-21:00],8400000000000004,True,38.530424,Setúbal,-8.885314,1512,Setúbal,Espaço navegante® Setúbal,10,"União das freguesias de Setúbal (São Julião, Nossa Senhora da Anunciada e Santa Maria da Graça)",210410400,2900-319,PT170,AML,Setúbal,"[160067, 160068, 160095, 162001, 162002, 162003, 162004, 162005, 162006, 162007, 162008, 162011]",https://www.carrismetropolitana.pt/encm
4,2,Praça Gomes Freire de Andrade 18,Espaço navegante®,2.000000,open,1,15,Setúbal,,100,ChIJ4YazSNI5GQ0R7Y2Fhiu0iek,[08:00-19:00],[08:00-19:00],[],,[],[08:00-19:00],[08:00-19:00],[08:00-19:00],8400000000000005,True,38.705786,Montijo,-8.976142,1507,Montijo,Espaço navegante® Montijo,10,União das freguesias de Montijo e Afonsoeiro,210410400,2870-237,PT170,AML,Montijo,"[100013, 100015, 100016, 100027, 100081, 100143]",https://www.carrismetropolitana.pt/encm
5,1,Av. Descobertas 90,Espaço navegante®,1.000000,open,0,11,Lisboa,,0,ChIJcYh0bA0tGQ0R8MrZS2YQXLE,[09:30-20:00],[09:30-20:00],[09:30-20:00],,[09:30-20:00],[09:30-20:00],[09:30-20:00],[09:30-20:00],8400000000000006,True,38.835582,Loures,-9.156227,1107,Loures,Espaço navegante® Loures,7,Loures,210410400,2670-457,PT170,AML,Loures,"[070383, 070385, 070435, 070437]",https://www.carrismetropolitana.pt/encm
6,1,Rua Elias Garcia 182B,Espaço navegante®,1.000000,open,0,11,Lisboa,,0,ChIJEc3j1VPNHg0Rk3GL2GCM9MQ,"[08:00-12:00, 14:00-18:00]","[08:00-12:00, 14:00-18:00]",[],,[],"[08:00-12:00, 14:00-18:00]","[08:00-12:00, 14:00-18:00]","[08:00-12:00, 14:00-18:00]",8400000000000007,True,38.758190,Amadora,-9.227082,1115,Amadora,Espaço navegante® Amadora (Elias Garcia),,,210410400,2700-315,PT170,AML,Amadora (Elias Garcia),"[030091, 030105, 030603, 030604, 030605, 030606, 030727, 030876]",https://www.carrismetropolitana.pt/encm
7,2,Avenida Dom Dinis 68A,Espaço navegante®,2.000000,open,0,11,Lisboa,,0,ChIJ3aFnf0MzGQ0RDZjWTfZH3bk,[09:00-20:00],[09:00-20:00],[],,[],[09:00-20:00],[09:00-20:00],[09:00-20:00],84000000000000

In [7]:
# ==========================================================================
# Carris Metropolitana API - Advanced Features & Real-time Data
# ==========================================================================

print_subheader("8. Real-Time Vehicle Positions")
df_vehicles, meta_vehicles = get_carris_data("vehicles")

if not df_vehicles.empty:
    print_success(f"Found {meta_vehicles.get('total', 0):,} active vehicles")

    # Show vehicle statistics
    print("\n📊 Vehicle Statistics:")
    print(f"   Total active vehicles: {len(df_vehicles):,}")

    if 'speed' in df_vehicles.columns:
        avg_speed = df_vehicles['speed'].mean()
        max_speed = df_vehicles['speed'].max()
        print(f"   Average speed: {avg_speed:.1f} km/h")
        print(f"   Max speed: {max_speed:.1f} km/h")

    # Show sample vehicles
    print_info("Sample active vehicles:")
    display_cols = [col for col in ['id', 'lat', 'lon', 'speed', 'heading', 'trip_id', 'pattern_id', 'timestamp'] if col in df_vehicles.columns]
    if display_cols:
        df_sample = df_vehicles[display_cols].head(5).copy()
        if 'timestamp' in df_sample.columns:
            df_sample['timestamp'] = df_sample['timestamp'].apply(format_timestamp)
        display(df_sample)
    else:
        display(df_vehicles.head(5))
else:
    print_info("No vehicle data available (vehicles may not be in service)")

# Test 9: Real-time arrivals for a specific stop
print_subheader("9. Real-Time Arrivals (Example Stop)")

# Get a sample stop from Lisbon (Praça do Comércio area)
sample_stop_id = "060001"  # A central Lisbon stop

df_realtime, meta_realtime = get_carris_stop_realtime(sample_stop_id)

if not df_realtime.empty:
    print_success(f"Real-time arrivals for stop {sample_stop_id}")
    print(f"   📍 Arrivals found: {meta_realtime.get('arrivals', 0)}")

    # Show arrivals
    display(df_realtime.head(10))
else:
    # Try another stop
    alternative_stop = "060003"
    print_info(f"No arrivals at {sample_stop_id}, trying {alternative_stop}...")
    df_realtime, meta_realtime = get_carris_stop_realtime(alternative_stop)
    if not df_realtime.empty:
        display(df_realtime.head(10))
    else:
        print_warning("No real-time arrivals available at the moment")

# Test 10: Pattern Details (Route Journey)
print_subheader("10. Pattern Details (Journey Information)")

# Get a sample pattern from lines data
if 'df_lines' in dir() and not df_lines.empty and 'patterns' in df_lines.columns:
    sample_line = df_lines.iloc[0]
    patterns = sample_line.get('patterns', [])

    if patterns and len(patterns) > 0:
        sample_pattern_id = patterns[0]
        print_info(f"Fetching pattern {sample_pattern_id} from line {sample_line.get('short_name', 'Unknown')}")

        pattern_data, meta_pattern = get_carris_pattern(sample_pattern_id)

        if pattern_data:
            # Handle if pattern_data is a list (take first element)
            if isinstance(pattern_data, list):
                pattern_data = pattern_data[0] if pattern_data else {}

            print_success("Pattern retrieved successfully")
            print("\n📋 Pattern Details:")
            print(f"   ID: {pattern_data.get('id', 'N/A')}")
            print(f"   Headsign: {pattern_data.get('headsign', 'N/A')}")
            print(f"   Stops in path: {len(pattern_data.get('path', []))}")
            print(f"   Number of trips: {len(pattern_data.get('trips', []))}")

            # Show first few stops in path
            path = pattern_data.get('path', [])
            if path:
                print("\n   First 5 stops in route:")
                for i, stop in enumerate(path[:5], 1):
                    print(f"      {i}. {stop.get('stop_name', 'Unknown')} ({stop.get('stop_id', 'N/A')})")
        else:
            print_warning("Could not fetch pattern details")
    else:
        print_info("No patterns available for the sample line")
else:
    print_info("Lines data not available for pattern lookup")

# Test 11: Shape Geometry
print_subheader("11. Shape Geometry (Route Coordinates)")

# Ensure pattern_data is a dict
if 'pattern_data' in dir() and pattern_data:
    if isinstance(pattern_data, list):
        pattern_data = pattern_data[0] if pattern_data else {}

    shape_id = pattern_data.get('shape_id') if isinstance(pattern_data, dict) else None

    if shape_id:
        print_info(f"Fetching shape {shape_id}")

        shape_data, meta_shape = get_carris_shape(shape_id)

        if shape_data:
            # Handle if shape_data is a list
            if isinstance(shape_data, list):
                shape_data = shape_data[0] if shape_data else {}

            print_success("Shape retrieved successfully")
            print("\n📐 Shape Details:")
            print(f"   ID: {shape_data.get('id', 'N/A')}")
            print(f"   Extension: {shape_data.get('extension', 'N/A')} meters")
            print(f"   Number of points: {len(shape_data.get('points', []))}")

            # GeoJSON info
            geojson = shape_data.get('geojson', {})
            if geojson:
                print(f"   GeoJSON type: {geojson.get('type', 'N/A')}")
                geometry = geojson.get('geometry', {})
                print(f"   Geometry type: {geometry.get('type', 'N/A')}")
                coords = geometry.get('coordinates', [])
                print(f"   Coordinate count: {len(coords)}")

                # Show first 3 coordinates
                if coords:
                    print("\n   First 3 coordinates [lon, lat]:")
                    for i, coord in enumerate(coords[:3], 1):
                        print(f"      {i}. {coord}")
        else:
            print_warning("Could not fetch shape details")
    else:
        print_info("No shape_id available in pattern data")
else:
    print_info("No pattern data available for shape lookup")

# ==========================================================================
# Carris API Summary
# ==========================================================================

print_subheader("📋 Carris Metropolitana API Summary")

# Calculate statistics
total_stops = meta_stops.get('total', 0) if 'meta_stops' in dir() else 'N/A'
total_lines = meta_lines.get('total', 0) if 'meta_lines' in dir() else 'N/A'
total_routes = meta_routes.get('total', 0) if 'meta_routes' in dir() else 'N/A'
total_vehicles = meta_vehicles.get('total', 0) if 'meta_vehicles' in dir() else 'N/A'
total_alerts = meta_alerts.get('total', 0) if 'meta_alerts' in dir() else 'N/A'

# Helper for aligned rows (67 chars inner content)
def carris_row(text):
    return f"│ {text:<67}│"

print("┌─────────────────────────────────────────────────────────────────────┐")
print(carris_row("🚌 Carris Metropolitana Network Statistics"))
print("├─────────────────────────────────────────────────────────────────────┤")
print(carris_row(f"Bus Stops:           {str(total_stops):>8}"))
print(carris_row(f"Bus Lines:           {str(total_lines):>8}"))
print(carris_row(f"Routes:              {str(total_routes):>8}"))
print(carris_row(f"Active Vehicles:     {str(total_vehicles):>8}"))
print(carris_row(f"Active Alerts:       {str(total_alerts):>8}"))
print("├─────────────────────────────────────────────────────────────────────┤")
print(carris_row("API Endpoints (v1 - primary):"))
print(carris_row("✅ /alerts           - Service disruptions"))
print(carris_row("✅ /vehicles         - Real-time positions"))
print(carris_row("✅ /stops            - Stop information + real-time"))
print(carris_row("✅ /lines            - Line metadata"))
print(carris_row("✅ /routes           - Route information"))
print(carris_row("✅ /patterns/:id     - Journey patterns"))
print(carris_row("✅ /shapes/:id       - Geographic shapes (GeoJSON)"))
print(carris_row("✅ /municipalities   - Municipality information (V1 only)"))
print(carris_row("✅ /datasets/facilities/encm - Customer service (V1 only)"))
print("├─────────────────────────────────────────────────────────────────────┤")
print(carris_row("Key Features:"))
print(carris_row("• Real-time vehicle tracking (GPS coordinates)"))
print(carris_row("• Real-time arrival predictions at stops"))
print(carris_row("• GeoJSON format for mapping integration"))
print(carris_row("• Comprehensive alert system for disruptions"))
print(carris_row("• Open API - no authentication required"))
print(carris_row("• /municipalities returned 23 API records in this run"))
print(carris_row("• Treat municipality count as endpoint metadata, not AML size"))
print("└─────────────────────────────────────────────────────────────────────┘")


8. Real-Time Vehicle Positions
--------------------------------------------------


✅ Found 1,690 active vehicles

📊 Vehicle Statistics:
   Total active vehicles: 1,690
   Average speed: 4.5 km/h
   Max speed: 28.1 km/h
ℹ️ Sample active vehicles:


,id,lat,lon,speed,trip_id,pattern_id,timestamp
0,|undefined,NaN,NaN,NaN,NaN,NaN,nan
1,41|300,38.767456,-9.298485,3.055556,[6KLCD]1612_0_2_0800_0829_0_1,1612_0_2,1970-01-21 14:11:32
2,41|306,38.741005,-9.239675,0.000000,[6KLCD]1014_0_2_1000_1029_0_1,1014_0_2,1970-01-21 14:11:38
3,41|308,38.758591,-9.244466,0.000000,[6KLCD]1717_0_2_1300_1329_1_1,1717_0_2,1970-01-21 14:11:48
4,41|311,38.782162,-9.302109,0.000000,[6KLCD]1715_0_2_1330_1359_0_1,1715_0_2,1970-01-21 14:11:48



9. Real-Time Arrivals (Example Stop)
--------------------------------------------------


✅ Real-time arrivals for stop 060001
   📍 Arrivals found: 146


,estimated_arrival,estimated_arrival_unix,headsign,line_id,observed_arrival,observed_arrival_unix,pattern_id,route_id,scheduled_arrival,scheduled_arrival_unix,stop_sequence,trip_id,vehicle_id
0,None,NaN,Vale da Amoreira,4701,None,NaN,4701_0_1,4701_0,04:00:00,1779073200,1,4701_0_1|1800|0400_undefined,None
1,None,NaN,Lisboa (Oriente),4701,None,NaN,4701_0_2,4701_0,05:20:00,1779078000,25,4701_0_2|1800|0430_undefined,None
2,None,NaN,Vale da Amoreira,4701,None,NaN,4701_0_1,4701_0,05:30:00,1779078600,1,4701_0_1|1800|0530_undefined,None
3,None,NaN,Lisboa (Oriente),4701,None,NaN,4701_0_2,4701_0,05:50:00,1779079800,25,4701_0_2|1800|0500_undefined,None
4,None,NaN,Vale da Amoreira,4701,None,NaN,4701_0_1,4701_0,06:00:00,1779080400,1,4701_0_1|1800|0600_undefined,None
5,None,NaN,Lisboa (Oriente),4701,None,NaN,4701_0_2,4701_0,06:20:00,1779081600,25,4701_0_2|1800|0530_undefined,None
6,None,NaN,Lisboa (Oriente),4701,None,NaN,4701_0_2,4701_0,06:30:00,1779082200,25,4701_0_2|1800|0540_undefined,None
7,None,NaN,Vale da Amoreira,4701,None,NaN,4701_0_1,4701_0,06:30:00,1779082200,1,4701_0_1|1800|0630_undefined,None
8,None,NaN,Lisboa (Oriente),4701,None,NaN,4701_0_2,4701_0,06:40:00,1779082800,25,4701_0_2|1800|0550_undefined,None
9,None,NaN,Vale da Amoreira,4701,None,NaN,4701_0_1,4701_0,06:45:00,1779083100,1,4701_0_1|1800|0645_undefined,None



10. Pattern Details (Journey Information)
--------------------------------------------------
ℹ️ Fetching pattern 1001_0_1 from line 1001
✅ Pattern retrieved successfully

📋 Pattern Details:
   ID: 1001_0_1
   Headsign: Reboleira (Estação)
   Stops in path: 23
   Number of trips: 256

   First 5 stops in route:
      1. Unknown (N/A)
      2. Unknown (N/A)
      3. Unknown (N/A)
      4. Unknown (N/A)
      5. Unknown (N/A)

11. Shape Geometry (Route Coordinates)
--------------------------------------------------
ℹ️ Fetching shape 1_A6RA0
✅ Shape retrieved successfully

📐 Shape Details:
   ID: 1_A6RA0
   Extension: 6578 meters
   Number of points: 961
   GeoJSON type: Feature
   Geometry type: LineString
   Coordinate count: 961

   First 3 coordinates [lon, lat]:
      1. [-9.220572, 38.734436]
      2. [-9.22055, 38.73442]
      3. [-9.22059, 38.73438]

📋 Carris Metropolitana API Summary
--------------------------------------------------
┌─────────────────────────────────────────────

### **🔌 Additional Carris Endpoints (Detail Queries)**

The following endpoints provide detailed information for specific resources and must be called with real identifiers:

| Endpoint | Description | Verified example |
|----------|-------------|------------------|
| `GET /stops/:id/realtime` | Realtime/scheduled arrivals at a stop | `/stops/060001/realtime` |
| `GET /patterns/:id` | Detailed pattern with path and trips | `/patterns/1612_0_2` |
| `GET /shapes/:id` | GeoJSON route geometry | `/shapes/1612_0_2` |
| `GET /stops/:id` | Single stop details | `/stops/060001` |
| `GET /lines/:id` | Single line details | `/lines/1612` |
| `GET /routes/:id` | Single route details | `/routes/1612_0` |

### **📊 Realtime Arrival Response Structure**

When querying `/stops/:id/realtime`, the response is a list of arrival records. The current API uses time strings plus Unix timestamp fields; realtime fields may be `null` when only scheduled data is available:

```json
{
  "line_id": "4001",
  "route_id": "4001_0",
  "pattern_id": "4001_0_3",
  "trip_id": "4001_0_3|1800|0710_undefined",
  "headsign": "Alcochete | Circular",
  "scheduled_arrival": "07:22:00",
  "scheduled_arrival_unix": 1779085320,
  "estimated_arrival": null,
  "estimated_arrival_unix": null,
  "observed_arrival": null,
  "vehicle_id": null
}
```

### **🗺️ GeoJSON Shape Response Structure**

When querying `/shapes/:id`, the response includes both point lists and a GeoJSON feature:

```json
{
  "id": "1612_0_2",
  "extension": 12500,
  "points": [{"shape_pt_lat": 38.72, "shape_pt_lon": -9.15, "shape_dist_traveled": 0}],
  "geojson": {
    "type": "Feature",
    "properties": {},
    "geometry": {
      "type": "LineString",
      "coordinates": [[-9.15, 38.72], [-9.16, 38.73]]
    }
  }
}
```

## <span style="color: #ffffff;">4. Carris Urban - GTFS & GTFS-RT</span>
<style>
@import url('https://fonts.cdnfonts.com/css/avenir-next-lt-pro?styles=29974');
</style>

<div style="background: transparent;
            padding: 10px; color: white; border-radius: 300px; text-align: center;
            border: 2px solid #FFCC00;">
    <center><h2 style="margin-left: 120px;margin-top: 10px; margin-bottom: 4px; color: #FFCC00;
                       font-size: 24px; font-family: 'Avenir Next LT Pro', sans-serif;"><b> 4. Carris Urban - GTFS & GTFS-RT</b></h2></center>
</div>

<br><br>

The **Carris Urban** data used by LISBOA covers the Lisbon urban Carris bus and tram network through:

1. **GTFS Static Data** - Pre-downloaded schedule/route information stored in a SQLite database
2. **GTFS-RT (Real-Time)** - Live vehicle positions via Protocol Buffers

**⚠️ IMPORTANT DISTINCTION:**
- **Carris Metropolitana** (Section 3): metropolitan/suburban bus API. The current API exposes 23 municipality records, but this is endpoint metadata and must not be described as "AML has 23 municipalities".
- **Carris Urban** (Section 4): Lisbon urban Carris network used for city buses and trams, including 12E, 15E, 18E, 24E, 25E, and 28E.

### **📊 Data Sources**

| Source | Type | URL | Format |
|--------|------|-----|--------|
| 🗄️ GTFS Static | Schedules, stops, routes | `https://gateway.carris.pt/gateway/gtfs/api/v2.8/GTFS` → local `data/carris/carris.db` | SQLite |
| 📡 GTFS-RT | Real-time vehicle positions | `https://gateway.carris.pt/gateway/gtfs/api/v2.8/GTFS/realtime/vehiclepositions` | Protocol Buffers |

### **🗃️ SQLite Database Schema**

The GTFS static data is stored locally in `data/carris/carris.db`. The row counts below are from the local snapshot used in this notebook execution and can change after a GTFS refresh:

<center><b>Table 4.1 | </b> GTFS <strong>agency</strong> - Transit agency information (1 row)<br>

|       | **Field**                          | **Description**                                                                 | **Type**     | **Used?** |
|:-----:|:----------------------------------:|:--------------------------------------------------------------------------------|:------------:|:---------:|
| **1** | `agency_id`                        | Unique identifier for the transit agency                                        | **`Text`**   | ✅ |
| **2** | `agency_name`                      | Full name of the transit agency                                                 | **`Text`**   | ✅ |
| **3** | `agency_url`                       | URL of the transit agency                                                       | **`Text`**   | ❌ |
| **4** | `agency_timezone`                  | Timezone of the agency (`Europe/Lisbon`)                                        | **`Text`**   | ✅ |
| **5** | `agency_lang`                      | Primary language                                                                | **`Text`**   | ❌ |
| **6** | `agency_phone`                     | Customer-service phone number                                                   | **`Text`**   | ❌ |
| **7** | `agency_fare_url`                  | Fare-information URL, if present                                                | **`Text`**   | ❌ |
| **8** | `agency_email`                     | Customer-service email, if present                                              | **`Text`**   | ❌ |

</center>

<br>

<center><b>Table 4.2 | </b> GTFS <strong>routes</strong> - Route/line definitions (174 rows)<br>

|       | **Field**                          | **Description**                                                                 | **Type**     | **Used?** |
|:-----:|:----------------------------------:|:--------------------------------------------------------------------------------|:------------:|:---------:|
| **1** | `route_id`                         | Unique route identifier                                                         | **`Text`**   | ✅ |
| **2** | `agency_id`                        | Reference to agency table                                                       | **`Text`**   | ✅ |
| **3** | `route_short_name`                 | Short route label, e.g. `28E`                                                   | **`Text`**   | ✅ |
| **4** | `route_long_name`                  | Long route name                                                                 | **`Text`**   | ✅ |
| **5** | `route_desc`                       | Optional route description                                                      | **`Text`**   | ❌ |
| **6** | `route_type`                       | GTFS route type (`0` tram, `3` bus, `800` trolleybus)                            | **`Integer`**| ✅ |
| **7** | `route_url`                        | Route information URL                                                           | **`Text`**   | ❌ |
| **8** | `route_color`                      | Route color                                                                     | **`Text`**   | ❌ |
| **9** | `route_text_color`                 | Text color for route labels                                                     | **`Text`**   | ❌ |

</center>

<br>

<center><b>Table 4.3 | </b> GTFS <strong>stops</strong> - Stop/station locations (2,337 rows)<br>

|       | **Field**                          | **Description**                                                                 | **Type**     | **Used?** |
|:-----:|:----------------------------------:|:--------------------------------------------------------------------------------|:------------:|:---------:|
| **1** | `stop_id`                          | Unique stop identifier, e.g. `4109` for Belém                                  | **`Text`**   | ✅ |
| **2** | `stop_code`                        | Public-facing stop code                                                         | **`Text`**   | ❌ |
| **3** | `stop_name`                        | Human-readable stop name                                                        | **`Text`**   | ✅ |
| **4** | `stop_desc`                        | Optional stop description                                                       | **`Text`**   | ❌ |
| **5** | `stop_lat`                         | WGS84 latitude coordinate                                                       | **`Numeric`**| ✅ |
| **6** | `stop_lon`                         | WGS84 longitude coordinate                                                      | **`Numeric`**| ✅ |
| **7** | `zone_id`                          | Fare zone identifier                                                            | **`Text`**   | ❌ |
| **8** | `stop_url`                         | Stop information URL                                                            | **`Text`**   | ❌ |
| **9** | `location_type`                    | GTFS location type                                                              | **`Integer`**| ❌ |
| **10**| `parent_station`                   | Parent station reference, if applicable                                         | **`Text`**   | ❌ |
| **11**| `stop_timezone`                    | Stop timezone, if provided                                                      | **`Text`**   | ❌ |
| **12**| `wheelchair_boarding`              | Accessibility flag                                                              | **`Integer`**| ❌ |

</center>

<br>

<center><b>Table 4.4 | </b> GTFS <strong>trips</strong> - Individual trip instances (77,164 rows)<br>

|       | **Field**                          | **Description**                                                                 | **Type**     | **Used?** |
|:-----:|:----------------------------------:|:--------------------------------------------------------------------------------|:------------:|:---------:|
| **1** | `trip_id`                          | Unique trip identifier                                                          | **`Text`**   | ✅ |
| **2** | `route_id`                         | Reference to routes table                                                       | **`Text`**   | ✅ |
| **3** | `service_id`                       | Reference to calendar/calendar_dates                                            | **`Text`**   | ✅ |
| **4** | `trip_headsign`                    | Destination sign displayed on vehicle                                           | **`Text`**   | ✅ |
| **5** | `trip_short_name`                  | Optional short trip identifier                                                  | **`Text`**   | ❌ |
| **6** | `direction_id`                     | Direction flag                                                                  | **`Integer`**| ✅ |
| **7** | `block_id`                         | Vehicle block identifier                                                        | **`Text`**   | ❌ |
| **8** | `shape_id`                         | Reference to shapes table                                                       | **`Text`**   | ✅ |
| **9** | `wheelchair_accessible`            | Accessibility flag                                                              | **`Integer`**| ❌ |
| **10**| `bikes_allowed`                    | Bicycle policy flag                                                             | **`Integer`**| ❌ |

</center>

<br>

<center><b>Table 4.5 | </b> GTFS <strong>stop_times</strong> - Arrival/departure times (2,187,780 rows)<br>

|       | **Field**                          | **Description**                                                                 | **Type**     | **Used?** |
|:-----:|:----------------------------------:|:--------------------------------------------------------------------------------|:------------:|:---------:|
| **1** | `trip_id`                          | Reference to trips table                                                        | **`Text`**   | ✅ |
| **2** | `arrival_time`                     | Scheduled arrival time (`HH:MM:SS`)                                             | **`Text`**   | ✅ |
| **3** | `departure_time`                   | Scheduled departure time (`HH:MM:SS`)                                           | **`Text`**   | ✅ |
| **4** | `stop_id`                          | Reference to stops table                                                        | **`Text`**   | ✅ |
| **5** | `stop_sequence`                    | Order of stop in trip                                                           | **`Integer`**| ✅ |
| **6** | `stop_headsign`                    | Override headsign for this stop                                                 | **`Text`**   | ❌ |
| **7** | `pickup_type`                      | Pickup availability                                                             | **`Integer`**| ❌ |
| **8** | `drop_off_type`                    | Drop-off availability                                                           | **`Integer`**| ❌ |
| **9** | `shape_dist_traveled`              | Distance along the shape                                                        | **`Numeric`**| ✅ |
| **10**| `timepoint`                        | Timepoint flag                                                                  | **`Integer`**| ❌ |

</center>

<br>

<center><b>Table 4.6 | </b> GTFS <strong>calendar</strong> - Regular service patterns (3 rows)<br>

|       | **Field**                          | **Description**                                                                 | **Type**     | **Used?** |
|:-----:|:----------------------------------:|:--------------------------------------------------------------------------------|:------------:|:---------:|
| **1** | `service_id`                       | Unique service identifier                                                       | **`Text`**   | ✅ |
| **2** | `monday` - `sunday`                | Binary flags for each weekday                                                   | **`Integer`**| ✅ |
| **3** | `start_date`                       | Service start date (`YYYYMMDD`)                                                 | **`Text`**   | ✅ |
| **4** | `end_date`                         | Service end date (`YYYYMMDD`)                                                   | **`Text`**   | ✅ |

</center>

<br>

<center><b>Table 4.7 | </b> GTFS <strong>calendar_dates</strong> - Service exceptions (40 rows)<br>

|       | **Field**                          | **Description**                                                                 | **Type**     | **Used?** |
|:-----:|:----------------------------------:|:--------------------------------------------------------------------------------|:------------:|:---------:|
| **1** | `service_id`                       | Reference to calendar table                                                     | **`Text`**   | ✅ |
| **2** | `date`                             | Exception date (`YYYYMMDD`)                                                     | **`Text`**   | ✅ |
| **3** | `exception_type`                   | `1` service added, `2` service removed                                          | **`Integer`**| ✅ |

</center>

<br>

<center><b>Table 4.8 | </b> GTFS <strong>shapes</strong> - Geographic path coordinates (142,048 rows)<br>

|       | **Field**                          | **Description**                                                                 | **Type**     | **Used?** |
|:-----:|:----------------------------------:|:--------------------------------------------------------------------------------|:------------:|:---------:|
| **1** | `shape_id`                         | Unique shape identifier                                                         | **`Text`**   | ✅ |
| **2** | `shape_pt_lat`                     | Latitude of shape point                                                         | **`Numeric`**| ✅ |
| **3** | `shape_pt_lon`                     | Longitude of shape point                                                        | **`Numeric`**| ✅ |
| **4** | `shape_pt_sequence`                | Order of point in shape                                                         | **`Integer`**| ✅ |
| **5** | `shape_dist_traveled`              | Distance travelled to this point                                                | **`Numeric`**| ✅ |

</center>

In [8]:
# ==========================================================================
# Carris Urban - GTFS SQLite Database Analysis
# ==========================================================================

import sqlite3
from pathlib import Path

def find_project_root(start: Path) -> Path:
    """Find the repository root from either notebook or repo execution paths."""
    for candidate in (start, *start.parents):
        if (candidate / "tools").exists() and (candidate / "data").exists():
            return candidate
    return start


# Database path: robust to execution from the notebook folder or repository root.
PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DB_PATH = PROJECT_ROOT / "data" / "carris" / "carris.db"

print_header("CARRIS GTFS DATABASE ANALYSIS", "🗃️")

if not DB_PATH.exists():
    print_error(f"Database not found: {DB_PATH.resolve()}")
else:
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    # Get all tables
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
    tables = [row[0] for row in cursor.fetchall()]

    print(f"\n📁 Database: {DB_PATH}")
    print(f"📊 Total Tables: {len(tables)}")
    print("─" * 75)

    # Table statistics
    table_stats = []
    for table in tables:
        cursor.execute(f"SELECT COUNT(*) FROM {table}")
        count = cursor.fetchone()[0]
        cursor.execute(f"PRAGMA table_info({table})")
        columns = cursor.fetchall()
        table_stats.append({
            "table": table,
            "rows": count,
            "columns": len(columns),
            "col_names": [col[1] for col in columns]
        })

    # Print summary table
    print(f"\n{'Table':<20} {'Rows':>12} {'Columns':>10}")
    print("─" * 45)
    for stat in table_stats:
        rows_fmt = f"{stat['rows']:,}"
        print(f"{stat['table']:<20} {rows_fmt:>12} {stat['columns']:>10}")
    print("─" * 45)

    total_rows = sum(s['rows'] for s in table_stats)
    total_fmt = f"{total_rows:,}"
    print(f"{'TOTAL':<20} {total_fmt:>12}")

    # Sample data from key tables
    print("\n" + "=" * 75)
    print_header("SAMPLE DATA FROM KEY TABLES", "📋")

    # 1. Agency
    print("\n\033[1m[agency]\033[0m - Transit agency information:")
    cursor.execute("SELECT * FROM agency LIMIT 2")
    for row in cursor.fetchall():
        print(f"  → {row}")

    # 2. Routes (sample trams and buses)
    print("\n\033[1m[routes]\033[0m - Iconic Lisbon trams:")
    cursor.execute("""
        SELECT route_id, route_short_name, route_long_name, route_type
        FROM routes
        WHERE route_short_name IN ('28E', '15E', '12E', '25E', '24E', '18E')
        ORDER BY route_short_name
    """)
    for row in cursor.fetchall():
        route_type_name = {0: "Tram 🚋", 3: "Bus 🚌", 800: "Trolleybus"}.get(row[3], "Unknown")
        print(f"  → {row[1]:>4} | {row[2]:<60} | {route_type_name}")

    # 3. Stops (tourist hotspots)
    print("\n\033[1m[stops]\033[0m - Popular tourist stops:")
    cursor.execute("""
        SELECT stop_id, stop_name, stop_lat, stop_lon
        FROM stops
        WHERE stop_name LIKE '%Belém%'
           OR stop_name LIKE '%Rossio%'
           OR stop_name LIKE '%Praça do Comércio%'
           OR stop_name LIKE '%Cais do Sodré%'
           OR stop_name LIKE '%Alfama%'
        LIMIT 8
    """)
    for row in cursor.fetchall():
        print(f"  → [{row[0]:>5}] {row[1]:<40} ({row[2]:.5f}, {row[3]:.5f})")

    # 4. Trips (28E example)
    print("\n\033[1m[trips]\033[0m - Sample trips for tram 28E:")
    cursor.execute("""
        SELECT t.trip_id, t.trip_headsign, t.direction_id, r.route_short_name
        FROM trips t
        JOIN routes r ON t.route_id = r.route_id
        WHERE r.route_short_name = '28E'
        LIMIT 4
    """)
    for row in cursor.fetchall():
        direction = "→" if row[2] == 0 else "←"
        trip_id_str = row[0][:30] if row[0] else "NULL"
        print(f"  → {row[3]} {direction} (trip: {trip_id_str}...)")

    # 5. Stop_times (schedule sample)
    print("\n\033[1m[stop_times]\033[0m - Schedule excerpt (28E at Chiado):")
    cursor.execute("""
        SELECT st.arrival_time, st.departure_time, s.stop_name, st.stop_sequence
        FROM stop_times st
        JOIN stops s ON st.stop_id = s.stop_id
        JOIN trips t ON st.trip_id = t.trip_id
        JOIN routes r ON t.route_id = r.route_id
        WHERE r.route_short_name = '28E' AND s.stop_name LIKE '%Chiado%'
        ORDER BY st.arrival_time
        LIMIT 6
    """)
    for row in cursor.fetchall():
        print(f"  → {row[0]} - {row[1]} | {row[2]:<30} (seq: {row[3]})")

    # 6. Calendar
    print("\n\033[1m[calendar]\033[0m - Service patterns:")
    cursor.execute("""
        SELECT service_id, monday, tuesday, wednesday, thursday, friday, saturday, sunday, start_date, end_date
        FROM calendar LIMIT 3
    """)
    for row in cursor.fetchall():
        days = "".join(["M" if row[1] else "_", "T" if row[2] else "_", "W" if row[3] else "_",
                        "T" if row[4] else "_", "F" if row[5] else "_", "S" if row[6] else "_", "S" if row[7] else "_"])
        print(f"  → {row[0][:20]:<20} | {days} | {row[8]} to {row[9]}")

    # 7. Route types breakdown
    print("\n\033[1m[Route Types Distribution]\033[0m:")
    cursor.execute("""
        SELECT route_type, COUNT(*) as count
        FROM routes GROUP BY route_type ORDER BY count DESC
    """)
    type_names = {0: "Tram 🚋", 3: "Bus 🚌", 800: "Trolleybus 🔌"}
    for row in cursor.fetchall():
        type_name = type_names.get(row[0], f"Type {row[0]}")
        print(f"  → {type_name:<20} {row[1]:>4} routes")

    conn.close()

    print("\n" + "─" * 75)
    print_success("Database analysis complete!")


🗃️ CARRIS GTFS DATABASE ANALYSIS

📁 Database: C:\Users\andre\OneDrive - NOVAIMS\[MDSAA-DS]_Thesis\LISBOA_MultiAgentSystem\data\carris\carris.db
📊 Total Tables: 9
───────────────────────────────────────────────────────────────────────────

Table                        Rows    Columns
─────────────────────────────────────────────
agency                          1          8
calendar                        3         10
calendar_dates                 40          3
routes                        174          9
shapes                    142,048          5
sqlite_stat1                   21          3
stop_times              2,187,780         10
stops                       2,337         12
trips                      77,164         10
─────────────────────────────────────────────
TOTAL                   2,409,568


📋 SAMPLE DATA FROM KEY TABLES

[agency] - Transit agency information:
  → ('1', 'Carris', 'http://www.carris.pt', 'Europe/Lisbon', 'pt', None, None, None)

[routes] - Iconic Lisbon t

  → 00:21:41 - 00:21:41 | Chiado                         (seq: 14)
  → 00:21:41 - 00:21:41 | Chiado                         (seq: 14)
  → 00:21:41 - 00:21:41 | Chiado                         (seq: 14)
  → 00:35:41 - 00:35:41 | Chiado                         (seq: 14)
  → 00:35:41 - 00:35:41 | Chiado                         (seq: 14)
  → 00:35:41 - 00:35:41 | Chiado                         (seq: 14)

[calendar] - Service patterns:
  → Inverno_Util_2026010 | MTWTF__ | 20260101 to 20260605
  → Inverno_DomingoFeria | ______S | 20260101 to 20260605
  → Inverno_Sabado_20260 | _____S_ | 20260101 to 20260605

[Route Types Distribution]:
  → Bus 🚌                 174 routes

───────────────────────────────────────────────────────────────────────────
✅ Database analysis complete!


### **🔗 Entity-Relationship Diagram**

The GTFS database follows a star-schema pattern where `stop_times` is the central fact table. The relationships below are logical GTFS relationships used by the notebook and tools; the SQLite import should not be interpreted as a fully enforced relational model:


<img src="../../data/carris/gtfs_er_diagram.png" alt="GTFS Schema Diagram" width="1000"/>

### **🔀 Key Relationships**

| Relationship | Cardinality | Description |
|--------------|-------------|-------------|
| `agency` → `routes` | 1:N | One agency operates multiple routes |
| `routes` → `trips` | 1:N | Each route has many scheduled trips |
| `calendar` → `trips` | 1:N | Service patterns define when trips run |
| `trips` → `stop_times` | 1:N | Each trip has ordered stops with times |
| `stops` → `stop_times` | 1:N | Each stop appears in many trips |
| `shapes` → `trips` | 1:N | Geographic paths for route visualization |
| `calendar` → `calendar_dates` | 1:N | Exceptions to regular service |

### **📡 GTFS-RT Real-Time Integration**

Real-time vehicle positions are fetched from the Protocol Buffers endpoint and enriched with static GTFS data:

In [9]:
# ==========================================================================
# Carris Urban - GTFS-RT Real-Time Vehicle Positions
# Endpoint: https://gateway.carris.pt/gateway/gtfs/api/v2.8/GTFS/realtime/vehiclepositions
# Format: Protocol Buffers (Google Transit Realtime specification)
# ==========================================================================

from datetime import datetime, timezone

import requests
from google.protobuf.json_format import MessageToDict
from google.transit import gtfs_realtime_pb2

# GTFS-RT Configuration
GTFS_RT_URL = "https://gateway.carris.pt/gateway/gtfs/api/v2.8/GTFS/realtime/vehiclepositions"

print_header("GTFS-RT REAL-TIME VEHICLE EXTRACTION", "📡")

try:
    # Fetch raw Protocol Buffers data
    response = requests.get(GTFS_RT_URL, timeout=15)
    response.raise_for_status()

    # Parse Protocol Buffers message
    feed = gtfs_realtime_pb2.FeedMessage()
    feed.ParseFromString(response.content)

    # Feed header info
    feed_timestamp = datetime.fromtimestamp(feed.header.timestamp, tz=timezone.utc)
    gtfs_version = feed.header.gtfs_realtime_version

    print(f"\n📅 Feed Timestamp: {feed_timestamp.strftime('%Y-%m-%d %H:%M:%S %Z')}")
    print(f"📊 GTFS-RT Version: {gtfs_version}")
    print(f"🚌 Total Vehicles: {len(feed.entity)}")
    print("─" * 75)

    # Show 2 raw examples before processing
    print("\n\033[1m[Raw GTFS-RT VehiclePosition Examples (Unprocessed)]\033[0m")
    print("─" * 75)
    count = 0
    for entity in feed.entity:
        if entity.HasField("vehicle") and count < 2:
            print(f"\n📄 Raw Example {count+1} (as received from Protocol Buffers):")
            print(json.dumps(MessageToDict(entity.vehicle), indent=2))
            count += 1

    # Extract vehicle data
    vehicles_raw = []
    for entity in feed.entity:
        if entity.HasField("vehicle"):
            v = entity.vehicle
            vehicles_raw.append({
                "vehicle_id": v.vehicle.id if v.vehicle.HasField("id") else None,
                "license_plate": v.vehicle.license_plate if v.vehicle.HasField("license_plate") else None,
                "trip_id": v.trip.trip_id if v.HasField("trip") and v.trip.HasField("trip_id") else None,
                "schedule_relationship": v.trip.ScheduleRelationship.Name(v.trip.schedule_relationship) if v.HasField("trip") and v.trip.HasField("schedule_relationship") else None,
                "route_id": v.trip.route_id if v.HasField("trip") and v.trip.HasField("route_id") else None,
                "direction_id": v.trip.direction_id if v.HasField("trip") and v.trip.HasField("direction_id") else None,
                "latitude": v.position.latitude if v.HasField("position") else None,
                "longitude": v.position.longitude if v.HasField("position") else None,
                "timestamp": datetime.fromtimestamp(v.timestamp, tz=timezone.utc) if v.HasField("timestamp") else None,
                "current_stop_sequence": v.current_stop_sequence if v.HasField("current_stop_sequence") else None,
                "stop_id": v.stop_id if v.HasField("stop_id") else None,
                "current_status": v.VehicleStopStatus.Name(v.current_status) if v.HasField("current_status") else None,
            })

    df_vehicles_rt = pd.DataFrame(vehicles_raw)

    # Join with routes table for route names
    conn = sqlite3.connect(DB_PATH)
    routes_df = pd.read_sql("SELECT route_id, route_short_name, route_long_name FROM routes", conn)
    conn.close()
    df_vehicles_rt = df_vehicles_rt.merge(routes_df, on='route_id', how='left')

    # Show raw Protocol Buffers fields available
    print("\n\033[1m[GTFS-RT VehiclePosition Fields]\033[0m")
    print("─" * 75)
    non_null_counts = df_vehicles_rt.notna().sum()
    for col in df_vehicles_rt.columns:
        fill_rate = (non_null_counts[col] / len(df_vehicles_rt)) * 100
        status = "✅" if fill_rate > 50 else "⚠️" if fill_rate > 0 else "❌"
        print(f"  {status} {col:<25} {non_null_counts[col]:>4}/{len(df_vehicles_rt)} ({fill_rate:5.1f}%)")

    # Sample vehicles (show first 8 vehicles)
    print("\n\033[1m[Sample: Real-Time Vehicle Positions]\033[0m")
    print("─" * 110)
    print(f"{'Route':<63} | {'Position':<22} | {'Status':<18} | {'Time'}")
    print("─" * 110)

    # Show first 8 vehicles regardless of type
    sample_vehicles = df_vehicles_rt.head(8)

    for _, v in sample_vehicles.iterrows():
        if pd.notna(v['route_short_name']):
            route = f"{v['route_id']} [{v['route_short_name']} - {v['route_long_name'] or 'N/A'}]"
        else:
            route = v['route_id'] or "N/A"
        lat = f"{v['latitude']:.5f}" if pd.notna(v['latitude']) else "N/A"
        lon = f"{v['longitude']:.5f}" if pd.notna(v['longitude']) else "N/A"
        position = f"({lat}, {lon})"
        status = v['current_status'] or "N/A"
        ts = v['timestamp'].strftime('%H:%M:%S') if pd.notna(v['timestamp']) else "N/A"
        print(f"  🚌 {route:<58} | {position:<22} | {status:<18} | {ts}")

    # Statistics by route
    print("\n\033[1m[Vehicles per Route (Top 15)]\033[0m")
    print("─" * 110)
    # Create combined route labels
    def get_route_label(row):
        if pd.notna(row['route_short_name']):
            return f"{row['route_id']} [{row['route_short_name']} - {row['route_long_name'] or 'N/A'}]"
        else:
            return row['route_id'] or "N/A"

    df_vehicles_rt['route_label'] = df_vehicles_rt.apply(get_route_label, axis=1)
    route_counts = df_vehicles_rt['route_label'].value_counts().head(15)
    for route, count in route_counts.items():
        bar = "█" * min(count, 30)
        print(f"  {str(route):<58} | {count:>3} | {bar}")

    # Status distribution
    print("\n\033[1m[Vehicle Status Distribution]\033[0m")
    status_map = {
        "IN_TRANSIT_TO": "🚌 In Transit",
        "STOPPED_AT": "🛑 Stopped",
        "INCOMING_AT": "➡️ Arriving"
    }
    status_counts = df_vehicles_rt['current_status'].value_counts()
    for status, count in status_counts.items():
        status_label = status_map.get(status, status)
        pct = (count / len(df_vehicles_rt)) * 100
        print(f"  {status_label:<20} {count:>4} ({pct:5.1f}%)")

    print("\n" + "─" * 75)
    print_success(f"Extracted {len(df_vehicles_rt)} vehicles at {feed_timestamp.strftime('%Y-%m-%d %H:%M:%S UTC')}")

except Exception as e:
    print_warning(f"GTFS-RT extraction unavailable in this run: {e}")


📡 GTFS-RT REAL-TIME VEHICLE EXTRACTION

📅 Feed Timestamp: 2026-05-18 12:44:58 UTC
📊 GTFS-RT Version: 2.0
🚌 Total Vehicles: 239
───────────────────────────────────────────────────────────────────────────

[Raw GTFS-RT VehiclePosition Examples (Unprocessed)]
───────────────────────────────────────────────────────────────────────────

📄 Raw Example 1 (as received from Protocol Buffers):
{
  "trip": {
    "tripId": "4454_20260101_103_0_5",
    "scheduleRelationship": "SCHEDULED",
    "routeId": "103_0",
    "directionId": 1
  },
  "position": {
    "latitude": 38.74732,
    "longitude": -9.14097
  },
  "currentStopSequence": 46,
  "currentStatus": "IN_TRANSIT_TO",
  "timestamp": "1779108267",
  "stopId": "202",
  "vehicle": {
    "id": "2797",
    "licensePlate": "AJ-57-LI"
  }
}

📄 Raw Example 2 (as received from Protocol Buffers):
{
  "trip": {
    "tripId": "8161_20260101_163_0_27",
    "scheduleRelationship": "SCHEDULED",
    "routeId": "163_0",
    "directionId": 1
  },
  "position":

### **🧪 Carris Urban - GTFS Tools Testing**

This section tests the **8 Carris Urban tools** currently exported by `tools/__init__.py`:

| Tool | Description | Input | Output |
|------|-------------|-------|--------|
| `carris_get_stops` | Search stops by name | query string, limit | List of stops with GPS coordinates |
| `carris_get_routes` | List routes or inspect a line | route_type, route_id, limit | Route information for trams and buses |
| `carris_get_arrivals` | Real-time arrivals | stop_id, limit | Next arrivals from GTFS-RT plus static schedule |
| `carris_get_next_departures` | Scheduled departures enriched with real-time when available | stop_id, start_time, route_short_name, limit | Upcoming departures at a stop |
| `carris_find_routes_between` | Routes between two locations | origin, destination, search_radius_km | Transport options with estimated times |
| `carris_get_realtime_vehicles` | GPS positions of vehicles | route_id, route_short_name, vehicle_type | Real-time vehicle positions |
| `carris_vehicle_eta` | Estimated time of arrival | route_short_name, stop_name | ETA calculated from GPS and schedule |
| `carris_get_service_frequency` | Average headway by time window | route_short_name, stop_name | Service frequency summary |

**Important notes:**
- Static data comes from the local SQLite database (`data/carris/carris.db`).
- Live data comes from the official Carris GTFS-RT vehicle feed when reachable.
- Nominatim/OpenStreetMap geocoding is used only where route planning needs named-place coordinates.
- Caching reduces repeated live calls while keeping the examples representative.


In [10]:
# ==========================================================================
# Carris Urban - GTFS Tools Testing (8 tools)
# ==========================================================================
from tools import (
    carris_find_routes_between,
    carris_get_arrivals,
    carris_get_next_departures,
    carris_get_realtime_vehicles,
    carris_get_routes,
    carris_get_service_frequency,
    carris_get_stops,
    carris_vehicle_eta,
 )

print_header("CARRIS URBAN - GTFS TOOLS TESTING", "🚋")

carris_sample_stop_id = "4109"
carris_sample_stop_name = "Belém"
carris_test_results = []


def show_tool_preview(result: str, max_lines: int = 14) -> None:
    lines = str(result).strip().split("\n")
    for line in lines[:max_lines]:
        print(f"   {line}")
    if len(lines) > max_lines:
        print(f"   ... ({len(lines) - max_lines} more lines)")


def record_carris_result(tool_name: str, success: bool, note: str) -> None:
    carris_test_results.append((tool_name, success, note))


print_subheader("1. carris_get_stops - Stop Search")
try:
    result = carris_get_stops.invoke({"query": carris_sample_stop_name, "limit": 5})
    show_tool_preview(result)
    record_carris_result("carris_get_stops", True, "Stop search by name")
except Exception as exc:
    print_error(f"Error: {exc}")
    record_carris_result("carris_get_stops", False, str(exc))

print_subheader("2. carris_get_routes - Route Listing")
try:
    result = carris_get_routes.invoke({"route_type": "tram", "limit": 8})
    show_tool_preview(result)
    record_carris_result("carris_get_routes", True, "Tram listing and route lookup")
except Exception as exc:
    print_error(f"Error: {exc}")
    record_carris_result("carris_get_routes", False, str(exc))

print_subheader("3. carris_get_next_departures - Scheduled Departures")
try:
    result = carris_get_next_departures.invoke({"stop_id": carris_sample_stop_id, "limit": 8})
    show_tool_preview(result)
    record_carris_result("carris_get_next_departures", True, "Scheduled departures at Belém")
except Exception as exc:
    print_error(f"Error: {exc}")
    record_carris_result("carris_get_next_departures", False, str(exc))

print_subheader("4. carris_get_arrivals - Real-Time Arrivals")
try:
    result = carris_get_arrivals.invoke({"stop_id": carris_sample_stop_id, "limit": 8})
    show_tool_preview(result)
    record_carris_result("carris_get_arrivals", True, "Real-time arrivals at Belém")
except Exception as exc:
    print_error(f"Error: {exc}")
    record_carris_result("carris_get_arrivals", False, str(exc))

print_subheader("5. carris_find_routes_between - Route Planning")
try:
    result = carris_find_routes_between.invoke({
        "origin": "Belém",
        "destination": "Praça do Comércio",
    })
    show_tool_preview(result)
    record_carris_result("carris_find_routes_between", True, "Belém to Praça do Comércio")
except Exception as exc:
    print_error(f"Error: {exc}")
    record_carris_result("carris_find_routes_between", False, str(exc))

print_subheader("6. carris_get_realtime_vehicles - GPS Tracking")
try:
    result = carris_get_realtime_vehicles.invoke({"route_id": "28E"})
    show_tool_preview(result, max_lines=16)
    record_carris_result("carris_get_realtime_vehicles", True, "28E vehicle positions")
except Exception as exc:
    print_error(f"Error: {exc}")
    record_carris_result("carris_get_realtime_vehicles", False, str(exc))

print_subheader("7. carris_vehicle_eta - ETA Calculation")
try:
    result = carris_vehicle_eta.invoke({
        "route_short_name": "15E",
        "stop_name": carris_sample_stop_name,
    })
    show_tool_preview(result)
    record_carris_result("carris_vehicle_eta", True, "ETA for 15E at Belém")
except Exception as exc:
    print_error(f"Error: {exc}")
    record_carris_result("carris_vehicle_eta", False, str(exc))

print_subheader("8. carris_get_service_frequency - Headway Summary")
try:
    result = carris_get_service_frequency.invoke({"route_short_name": "28E"})
    show_tool_preview(result)
    record_carris_result("carris_get_service_frequency", True, "Frequency summary for 28E")
except Exception as exc:
    print_error(f"Error: {exc}")
    record_carris_result("carris_get_service_frequency", False, str(exc))

passed = sum(1 for _, success, _ in carris_test_results if success)
failed = len(carris_test_results) - passed

print_subheader("Carris Urban Summary")
for tool_name, success, note in carris_test_results:
    status = "✅ PASS" if success else "❌ FAIL"
    print(f"   {status} | {tool_name} | {note}")

if failed:
    print_warning(f"Carris Urban tool smoke test finished with {failed} failing tool(s).")
else:
    print_success("All 8 Carris Urban tools completed successfully.")


🚋 CARRIS URBAN - GTFS TOOLS TESTING

1. carris_get_stops - Stop Search
--------------------------------------------------
   ### 🚌 **Carris stops near 'Belém'** (5 shown)
   
   **🚏 Av. Torre Belém**
       - 🆔 **Stop ID:** `13213`
       - 🔢 **Stop code:** `13213`
       - 📍 [Open map](https://www.google.com/maps/search/?api=1&query=38.69687%2C-9.21457)
   
   **🚏 Belém**
       - 🆔 **Stop ID:** `4111`
       - 🔢 **Stop code:** `4111`
       - 📍 [Open map](https://www.google.com/maps/search/?api=1&query=38.69712%2C-9.20083)
   
   **🚏 Belém**
       - 🆔 **Stop ID:** `4109`
   ... (12 more lines)

2. carris_get_routes - Route Listing
--------------------------------------------------
   Rotas Carris (Lisboa Urbano)
   
   ELÉTRICOS
   ------------------------------
      12E: Martim Moniz - Pç. Luis Camões
      15E: Pç. Figueira - Algés
      18E: Cais Sodré - Cemitério Ajuda
      24E: Pç. Luis Camões - Campolide
      25E: Pç Figueira - Campo Ourique (Prazeres)
      28E: Martim Mo

   🚌 **Next Departures from Belém**  
      (📡 Real-Time Data Active)  
      📡 Carris GTFS-RT: live vehicle feed active.  
   ---  
   📍 **[729] Para B. Padre Cruz**  
      🕒 **13:54** (8m late), 14:03  
   
   📍 **[728] Para Portela**  
      🕒 13:52, 14:08  
   
   📍 **[727] Para Estação Roma-Areeiro**  
      🕒 13:53, 14:06  
   
   📍 **[751] Para Estação Campolide**  
   ... (3 more lines)

4. carris_get_arrivals - Real-Time Arrivals
--------------------------------------------------


   Próximas Chegadas: Belém
      ID: 4109 | Atualizado: 13:45
   
   📡 Carris GTFS-RT: cached live snapshot in use (0s old).
   
   [SCHEDULE] Autocarro 728 -> Restelo - Portela
      Hora: 13:52
   
   [SCHEDULE] Autocarro 727 -> Estação Roma-Areeiro - Restelo
      Hora: 13:53
   
   [REAL-TIME] Autocarro 729 -> B. Padre Cruz - Algés
      Hora: 13:54 (8 min late)
   ... (22 more lines)

5. carris_find_routes_between - Route Planning
--------------------------------------------------


   Routes: Belém -> Praça do Comércio
   
      From: Belém
      To: Praça do Comércio
   
   ✅ **Direct routes found:** 2
   
   📡 Carris GTFS-RT: cached live snapshot in use (0s old).
   
   TRAMS
   ----------------------------------------
      15E: para Pç. Figueira
        Stops: board at Belém (Museu Coches); leave at stop Pç. Comércio.
   ... (11 more lines)

6. carris_get_realtime_vehicles - GPS Tracking
--------------------------------------------------
   Veículos Carris em Tempo Real
   Dados de: 13:45:08
   
   📡 Carris GTFS-RT: cached live snapshot in use (19s old).
   
   ELÉTRICOS
   ----------------------------------------
   28E -> Martim Moniz [Em trânsito]
      GPS: 38.71393, -9.16138 | Próxima paragem: Estrela - R. Domingos Seque…
   28E -> Martim Moniz [Em trânsito]
      GPS: 38.71272, -9.15718
   28E -> Martim Moniz [Em trânsito]
      GPS: 38.70838, -9.13975 | Próxima paragem: Lg. Academia Nacional Belas…
   28E -> Martim Moniz [Em trânsito]
      GPS: 38.710

   ### 🚋 **15E at Av. Torre Belém**
   
   - 🕒 **Updated:** 13:45
   - 📡 Carris GTFS-RT: cached live snapshot in use (19s old).
   - ℹ️ **Live Arrival Estimate:** no active vehicle is currently matched to this stop.
   - 🕒 **Scheduled fallback:**

8. carris_get_service_frequency - Headway Summary
--------------------------------------------------


   ### 🚋 **Route 28E service frequency**
   
   - 📍 **Stop:** Campo Ourique (Prazeres)
   - 📊 **Total departures today (Paragens):** 226
   
   **🌅 Morning (06:00-09:59)**
       - ⏱️ **Avg frequency:** 6 min · **Min/Max:** 4-10 min
       - 🕒 **First:** 06:25 · **Last:** 09:59 · 🚋 **Departures:** 38
   
   **☀️ Midday (10:00-13:59)**
       - ⏱️ **Avg frequency:** 4 min · **Min/Max:** 3-5 min
       - 🕒 **First:** 10:03 · **Last:** 13:56 · 🚋 **Departures:** 55
   
   **🌤️ Afternoon (14:00-17:59)**
   ... (12 more lines)

Carris Urban Summary
--------------------------------------------------
   ✅ PASS | carris_get_stops | Stop search by name
   ✅ PASS | carris_get_routes | Tram listing and route lookup
   ✅ PASS | carris_get_next_departures | Scheduled departures at Belém
   ✅ PASS | carris_get_arrivals | Real-time arrivals at Belém
   ✅ PASS | carris_find_routes_between | Belém to Praça do Comércio
   ✅ PASS | carris_get_realtime_vehicles | 28E vehicle positions
   ✅ PASS | carris_ve

In [11]:
# ==========================================================================
# Carris Urban - Current Tool Signatures and Usage Notes
# ==========================================================================
print_header("CARRIS URBAN - CURRENT TOOL SNAPSHOT", "🧭")
print_info("This compact cell replaces the older duplicate Carris-only smoke suite.")
print_info("For full exported-tool coverage, use the final runtime tool audit cell at the end of the notebook.")

carris_tool_snapshot = pd.DataFrame(
    [
        {"tool": "carris_get_stops", "signature_focus": "query, limit", "main_use": "Find stops near a landmark or neighbourhood"},
        {"tool": "carris_get_routes", "signature_focus": "route_type, route_id, limit", "main_use": "Inspect tram and bus routes"},
        {"tool": "carris_get_arrivals", "signature_focus": "stop_id, limit", "main_use": "Check near-term arrivals with GTFS-RT enrichment"},
        {"tool": "carris_get_next_departures", "signature_focus": "stop_id, start_time, route_short_name, limit", "main_use": "See scheduled departures at a stop"},
        {"tool": "carris_find_routes_between", "signature_focus": "origin, destination, search_radius_km", "main_use": "Plan a direct Carris-only journey"},
        {"tool": "carris_get_realtime_vehicles", "signature_focus": "route_id, route_short_name, vehicle_type", "main_use": "Track buses or trams in real time"},
        {"tool": "carris_vehicle_eta", "signature_focus": "route_short_name, stop_name", "main_use": "Estimate vehicle arrival to a stop"},
        {"tool": "carris_get_service_frequency", "signature_focus": "route_short_name, stop_name", "main_use": "Estimate headway by time window"},
    ]
)
display(carris_tool_snapshot)

print_info("The Belém, Rossio, Praça do Comércio, and 28E examples used in this notebook are kept because they exercise common tourist-facing scenarios in the current LISBOA runtime.")


🧭 CARRIS URBAN - CURRENT TOOL SNAPSHOT
ℹ️ This compact cell replaces the older duplicate Carris-only smoke suite.
ℹ️ For full exported-tool coverage, use the final runtime tool audit cell at the end of the notebook.


,tool,signature_focus,main_use
0,carris_get_stops,"query, limit",Find stops near a landmark or neighbourhood
1,carris_get_routes,"route_type, route_id, limit",Inspect tram and bus routes
2,carris_get_arrivals,"stop_id, limit",Check near-term arrivals with GTFS-RT enrichment
3,carris_get_next_departures,"stop_id, start_time, route_short_name, limit",See scheduled departures at a stop
4,carris_find_routes_between,"origin, destination, search_radius_km",Plan a direct Carris-only journey
5,carris_get_realtime_vehicles,"route_id, route_short_name, vehicle_type",Track buses or trams in real time
6,carris_vehicle_eta,"route_short_name, stop_name",Estimate vehicle arrival to a stop
7,carris_get_service_frequency,"route_short_name, stop_name",Estimate headway by time window


ℹ️ The Belém, Rossio, Praça do Comércio, and 28E examples used in this notebook are kept because they exercise common tourist-facing scenarios in the current LISBOA runtime.


## <span style="color: #ffffff;">5. Comboios de Portugal (CP) API</span>
<style>
@import url('https://fonts.cdnfonts.com/css/avenir-next-lt-pro?styles=29974');
</style>

<div style="background: transparent;
            padding: 10px; color: white; border-radius: 300px; text-align: center;
            border: 2px solid #F58228;">
    <center><h2 style="margin-left: 120px;margin-top: 10px; margin-bottom: 4px; color: #F58228;
                       font-size: 24px; font-family: 'Avenir Next LT Pro', sans-serif;"><b> 5. Comboios de Portugal (CP) API</b></h2></center>
</div>

<br><br>

The LISBOA CP runtime combines two source families:

- **Official CP GTFS static schedules:** `https://publico.cp.pt/gtfs/gtfs.zip`
- **Community live status API:** `https://comboios.live/api/stations` and `https://comboios.live/api/vehicles`

The notebook cell below explores the live `comboios.live` station and vehicle endpoints. The production tools also use the local CP GTFS database for schedule-based station, route, trip-planning, and frequency logic.

> **Scope note:** LISBOA treats this module as **AML suburban CP rail**. Fertagus is a separate rail operator and must not be described as a CP line. Long-distance national rail planning is outside the implemented CP tool scope.

### **📡 Data Sources and Endpoints**

<center><b>Table 5.0 | </b> CP-related sources used by LISBOA. <br>

| Source | Method | Description | Role in LISBOA |
|--------|--------|-------------|----------------|
| `https://publico.cp.pt/gtfs/gtfs.zip` | GET | Official CP GTFS static feed | Static schedules, routes, stops and trip planning |
| `https://comboios.live/api/stations` | GET | Community station feed with coordinates | Live station lookup and AML filtering |
| `https://comboios.live/api/vehicles` | GET | Community vehicle feed with train positions and delays | Live/near-live AML status summaries |

</center>
<br>

### **🗺️ AML Bounding Box**

For notebook exploration, stations and vehicles are filtered to the Lisbon Metropolitan Area using:
- **Latitude:** 38.5° to 39.1° N
- **Longitude:** -9.5° to -8.8° W
- **Current execution:** the live station filter returned **72 AML stations**. This count is snapshot-dependent and may differ from older comments in code or future API responses.

### **🚆 Supported CP Suburban Lines for Tourism**

<center><b>Table 5.1 | </b> CP suburban lines in LISBOA scope. <br>

| Line | Route | Tourism relevance | Frequency note |
|------|-------|-------------------|----------------|
| 🌊 **Cascais** | Cais do Sodré ↔ Cascais | Coastal towns and beaches, Estoril | Timetable-dependent frequent suburban service |
| 🏰 **Sintra** | Rossio / Oriente ↔ Sintra | Sintra tourism corridor | Timetable-dependent frequent suburban service |
| 🌾 **Azambuja** | Santa Apolónia / Oriente ↔ Azambuja | North-suburban corridor, Oriente | Timetable-dependent suburban service |
| 🌉 **Sado** | Barreiro ↔ Setúbal | South-bank CP suburban corridor | Timetable-dependent suburban service |

</center>

### **🔎 Dataset Attributes**

<center><b>Table 5.2 | </b> CP <strong>Stations</strong> Attributes from the live station feed. <br>

<br>

|       | **Field**                          | **Description**                                                                 | **Type**     |
|:-----:|:----------------------------------:|:--------------------------------------------------------------------------------|:-----------:|
| **1** | `code`                             | Station code                                                                    | **`Text`**   |
| **2** | `designation`                      | Full station name                                                               | **`Text`**   |
| **3** | `latitude`                         | GPS latitude (WGS84)                                                            | **`Numeric`**|
| **4** | `longitude`                        | GPS longitude (WGS84)                                                           | **`Numeric`**|
| **5** | `region`                           | Geographic region                                                               | **`Text`**   |
| **6** | `railways`                         | Array of railway lines serving the station                                      | **`List`**   |

</center>

<br>

<center><b>Table 5.3 | </b> CP <strong>Vehicles</strong> Attributes from the live vehicle feed. <br>

<br>

|       | **Field**                          | **Description**                                                                 | **Type**     |
|:-----:|:----------------------------------:|:--------------------------------------------------------------------------------|:-----------:|
| **1** | `trainNumber`                      | Train service number                                                            | **`Text`**   |
| **2** | `runDate`                          | Date of the train service                                                       | **`Date`**   |
| **3** | `delay`                            | Current delay in seconds (0 = on time, positive = late)                         | **`Numeric`**|
| **4** | `status`                           | Train status, when provided by the source                                       | **`Text`**   |
| **5** | `latitude` / `longitude`           | Current GPS position                                                            | **`Numeric`**|
| **6** | `lastStation`                      | Last station the train passed                                                   | **`Text`**   |
| **7** | `origin`                           | Origin station name                                                             | **`Text`**   |
| **8** | `destination`                      | Destination station name                                                        | **`Text`**   |
| **9** | `service`                          | Service type, when exposed by the source                                        | **`Text`**   |
| **10**| `hasDisruptions`                   | Boolean flag for service disruptions                                            | **`Boolean`**|

</center>

<br>

### **⏱️ Delay Interpretation**

<center><b>Table 5.4 | </b> CP Train <strong>Delay Interpretation</strong>. <br>

| Delay (seconds) | Status | Interpretation |
|-----------------|--------|----------------|
| 0 or negative | ✅ On time | Train is running on schedule |
| 1-300 | ⚠️ Minor delay | Up to 5 minutes late |
| 301-900 | ⚠️ Moderate delay | 5-15 minutes late |
| 901+ | 🔴 Significant delay | More than 15 minutes late |

</center>

<br><br>

In [12]:
# ==========================================================================
# CP (Comboios de Portugal) API Configuration
# ==========================================================================

# Static schedule source and live endpoints.
CP_GTFS_URL = "https://publico.cp.pt/gtfs/gtfs.zip"
CP_STATIONS_URL = "https://comboios.live/api/stations"
CP_VEHICLES_URL = "https://comboios.live/api/vehicles"

# AML (Área Metropolitana de Lisboa) Bounding Box
# Used to filter stations and trains to the Lisbon metropolitan area
AML_BOUNDS = {
    "lat_min": 38.5,
    "lat_max": 39.1,
    "lon_min": -9.5,
    "lon_max": -8.8
}

# CP suburban lines in LISBOA scope
CP_LINES = {
    "cascais": {
        "name": "Linha de Cascais",
        "emoji": "🌊",
        "color": "#0088CC",
        "description": "Cais do Sodré ↔ Cascais (coastal route)",
        "terminus": ["Cais do Sodré", "Cascais"],
        "frequency": "~20 min"
    },
    "sintra": {
        "name": "Linha de Sintra",
        "emoji": "🏰",
        "color": "#FF6600",
        "description": "Rossio/Oriente ↔ Sintra",
        "terminus": ["Rossio", "Sintra"],
        "frequency": "~20 min"
    },
    "azambuja": {
        "name": "Linha de Azambuja",
        "emoji": "🌾",
        "color": "#009933",
        "description": "Santa Apolónia/Oriente ↔ Azambuja",
        "terminus": ["Santa Apolónia", "Azambuja"],
        "frequency": "~30 min"
    },
    "sado": {
        "name": "Linha do Sado",
        "emoji": "🌉",
        "color": "#9933FF",
        "description": "Barreiro ↔ Setúbal (south-bank CP suburban line)",
        "terminus": ["Barreiro", "Setúbal"],
        "frequency": "Variable"
    }
}

# Train status translations
TRAIN_STATUS = {
    "IN_TRANSIT": {"emoji": "🚆", "desc": "In transit"},
    "AT_STOP": {"emoji": "🛑", "desc": "At station"},
    "DEPARTED": {"emoji": "✅", "desc": "Departed"},
    "ARRIVED": {"emoji": "🏁", "desc": "Arrived"},
    "CANCELLED": {"emoji": "❌", "desc": "Cancelled"},
    "DELAYED": {"emoji": "⏰", "desc": "Delayed"},
}

# ==========================================================================
# CP API Functions
# ==========================================================================

def get_cp_stations(filter_aml: bool = True) -> Tuple[pd.DataFrame, Dict]:
    """
    Fetches CP train stations.

    Args:
        filter_aml (bool): If True, filter to AML region only.

    Returns:
        Tuple[pd.DataFrame, Dict]: DataFrame with stations and metadata.
    """
    metadata = {"url": CP_STATIONS_URL, "status": "pending", "filter_aml": filter_aml}

    try:
        response = requests.get(CP_STATIONS_URL, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        data = response.json()

        metadata["status"] = "success"

        if 'stations' in data:
            df = pd.DataFrame(data['stations'])
            metadata["total"] = len(df)

            # Convert coordinates to numeric
            if 'latitude' in df.columns:
                df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
            if 'longitude' in df.columns:
                df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')

            if filter_aml and 'latitude' in df.columns and 'longitude' in df.columns:
                # Filter to AML bounding box
                df_aml = df[
                    (df['latitude'] >= AML_BOUNDS['lat_min']) &
                    (df['latitude'] <= AML_BOUNDS['lat_max']) &
                    (df['longitude'] >= AML_BOUNDS['lon_min']) &
                    (df['longitude'] <= AML_BOUNDS['lon_max'])
                ].copy()
                metadata["aml_stations"] = len(df_aml)
                return df_aml, metadata

            return df, metadata
        else:
            metadata["error"] = "No stations in response"
            return pd.DataFrame(), metadata

    except requests.exceptions.RequestException as e:
        metadata["status"] = "error"
        metadata["error"] = str(e)
        print_error(f"Error fetching CP stations: {e}")
        return pd.DataFrame(), metadata


def get_cp_vehicles(filter_aml: bool = True) -> Tuple[pd.DataFrame, Dict]:
    """
    Fetches real-time CP train positions.

    Args:
        filter_aml (bool): If True, filter to AML region only.

    Returns:
        Tuple[pd.DataFrame, Dict]: DataFrame with vehicles and metadata.
    """
    metadata = {"url": CP_VEHICLES_URL, "status": "pending", "filter_aml": filter_aml}

    try:
        response = requests.get(CP_VEHICLES_URL, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        data = response.json()

        metadata["status"] = "success"

        if 'vehicles' in data:
            df = pd.DataFrame(data['vehicles'])
            metadata["total"] = len(df)

            # Convert coordinates to numeric
            if 'latitude' in df.columns:
                df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
            if 'longitude' in df.columns:
                df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')
            if 'delay' in df.columns:
                df['delay'] = pd.to_numeric(df['delay'], errors='coerce')

            if filter_aml and 'latitude' in df.columns and 'longitude' in df.columns:
                # Filter to AML bounding box (with some buffer for trains approaching)
                df_aml = df[
                    (df['latitude'] >= AML_BOUNDS['lat_min'] - 0.1) &
                    (df['latitude'] <= AML_BOUNDS['lat_max'] + 0.1) &
                    (df['longitude'] >= AML_BOUNDS['lon_min'] - 0.1) &
                    (df['longitude'] <= AML_BOUNDS['lon_max'] + 0.1)
                ].copy()
                metadata["aml_vehicles"] = len(df_aml)
                return df_aml, metadata

            return df, metadata
        else:
            metadata["error"] = "No vehicles in response"
            return pd.DataFrame(), metadata

    except requests.exceptions.RequestException as e:
        metadata["status"] = "error"
        metadata["error"] = str(e)
        print_error(f"Error fetching CP vehicles: {e}")
        return pd.DataFrame(), metadata


def format_delay(delay_seconds) -> str:
    """Formats delay in seconds to human-readable string."""
    if delay_seconds is None or pd.isna(delay_seconds):
        return "On time ✅"
    try:
        delay = int(delay_seconds)
    except (ValueError, TypeError):
        return "On time ✅"
    if delay <= 0:
        return "On time ✅"
    elif delay < 60:
        return f"{delay}s late ⚠️"
    elif delay < 3600:
        return f"{delay // 60}m late ⚠️"
    else:
        return f"{delay // 3600}h {(delay % 3600) // 60}m late 🔴"


# ==========================================================================
# CP API Testing & Visualization
# ==========================================================================

print_header("CP Comboios de Portugal API - Train Network", "🚆")

# Test 1: API Connectivity
print_subheader("1. API Connectivity Test")

for name, url in [("Stations", CP_STATIONS_URL), ("Vehicles", CP_VEHICLES_URL)]:
    start_time = time.time()
    try:
        response = requests.get(url, timeout=REQUEST_TIMEOUT)
        elapsed = time.time() - start_time
        if response.status_code == 200:
            print_success(f"{name} API responding (Status: {response.status_code}, Time: {elapsed:.2f}s)")
        else:
            print_error(f"{name} API error (Status: {response.status_code})")
    except Exception as e:
        print_error(f"{name} API connection failed: {e}")

# Test 2: All Stations
print_subheader("2. Train Stations (All Portugal)")
df_stations_all, meta_stations_all = get_cp_stations(filter_aml=False)

if not df_stations_all.empty:
    print_success(f"Total CP stations: {meta_stations_all.get('total', 0)}")
    print(f"   Available columns: {list(df_stations_all.columns)}")
else:
    print_error("Could not fetch stations")

# Test 3: AML Stations
print_subheader("3. Train Stations (Lisbon Metropolitan Area)")
df_stations_aml, meta_stations_aml = get_cp_stations(filter_aml=True)

if not df_stations_aml.empty:
    print_success(f"Stations in AML: {meta_stations_aml.get('aml_stations', 0)}")
    print_info(f"AML Bounding Box: Lat [{AML_BOUNDS['lat_min']}, {AML_BOUNDS['lat_max']}], Lon [{AML_BOUNDS['lon_min']}, {AML_BOUNDS['lon_max']}]")

    # Display AML stations
    display_cols = [col for col in ['code', 'designation', 'latitude', 'longitude'] if col in df_stations_aml.columns]
    if display_cols:
        display(df_stations_aml[display_cols].head(15))
    else:
        display(df_stations_aml.head(15))
else:
    print_warning("No stations found in AML region")

# Test 4: Real-time Vehicles (All)
print_subheader("4. Real-Time Trains (All Portugal)")
df_vehicles_all, meta_vehicles_all = get_cp_vehicles(filter_aml=False)

if not df_vehicles_all.empty:
    print_success(f"Total active trains: {meta_vehicles_all.get('total', 0)}")
    print(f"   Available columns: {list(df_vehicles_all.columns)}")

    # Calculate delay statistics
    if 'delay' in df_vehicles_all.columns:
        delayed = df_vehicles_all[df_vehicles_all['delay'] > 0]
        print("\n   📊 Delay Statistics:")
        print(f"      Trains with delays: {len(delayed)} ({len(delayed)/len(df_vehicles_all)*100:.1f}%)")
        if not delayed.empty:
            print(f"      Average delay: {delayed['delay'].mean():.0f} seconds")
            print(f"      Max delay: {delayed['delay'].max():.0f} seconds")
else:
    print_warning("No active trains data available")

# Test 5: AML Vehicles
print_subheader("5. Real-Time Trains (Lisbon Metropolitan Area)")
df_vehicles_aml, meta_vehicles_aml = get_cp_vehicles(filter_aml=True)

if not df_vehicles_aml.empty:
    print_success(f"Active trains in AML: {meta_vehicles_aml.get('aml_vehicles', 0)}")

    # Add delay formatting
    if 'delay' in df_vehicles_aml.columns:
        df_vehicles_aml['delay_formatted'] = df_vehicles_aml['delay'].apply(format_delay)

    if 'status' in df_vehicles_aml.columns:
        df_vehicles_aml['status_emoji'] = df_vehicles_aml['status'].map(
            lambda x: TRAIN_STATUS.get(x, {}).get('emoji', '❓') if pd.notna(x) else '❓'
        )

    # Display AML trains
    display_cols = [col for col in ['trainNumber', 'status_emoji', 'status', 'delay_formatted', 'latitude', 'longitude']
                   if col in df_vehicles_aml.columns]
    if display_cols:
        display(df_vehicles_aml[display_cols].head(15))
    else:
        display(df_vehicles_aml.head(15))
else:
    print_warning("No active trains in AML region")

# Test 6: Line Information
print_subheader("6. CP Lines Serving Lisbon")

print("\n📋 Train Lines in the Lisbon Metropolitan Area:\n")
for line_key, line_info in CP_LINES.items():
    print(f"   {line_info['emoji']} {line_info['name']}")
    print(f"      {line_info['description']}")
    print(f"      Frequency: {line_info['frequency']}")
    print()

# ==========================================================================
# CP API Summary
# ==========================================================================

print_subheader("📋 CP Comboios de Portugal API Summary")

total_stations = meta_stations_all.get('total', 'N/A') if 'meta_stations_all' in dir() else 'N/A'
aml_stations = meta_stations_aml.get('aml_stations', 'N/A') if 'meta_stations_aml' in dir() else 'N/A'
total_vehicles = meta_vehicles_all.get('total', 'N/A') if 'meta_vehicles_all' in dir() else 'N/A'
aml_vehicles = meta_vehicles_aml.get('aml_vehicles', 'N/A') if 'meta_vehicles_aml' in dir() else 'N/A'

# Helper for aligned rows (67 chars inner content)
def cp_row(text):
    return f"│ {text:<67}│"

print("┌─────────────────────────────────────────────────────────────────────┐")
print(cp_row("🚆 CP Train Network Statistics"))
print("├─────────────────────────────────────────────────────────────────────┤")
print(cp_row(f"Total Stations (Portugal):   {str(total_stations):>6}"))
print(cp_row(f"Stations in AML:             {str(aml_stations):>6}"))
print(cp_row(f"Active Trains (Portugal):    {str(total_vehicles):>6}"))
print(cp_row(f"Active Trains in AML:        {str(aml_vehicles):>6}"))
print("├─────────────────────────────────────────────────────────────────────┤")
print(cp_row("Data sources:"))
print(cp_row("✅ CP GTFS zip       - Official static schedules"))
print(cp_row("✅ /stations         - comboios.live station list"))
print(cp_row("✅ /vehicles         - comboios.live train positions & delays"))
print("├─────────────────────────────────────────────────────────────────────┤")
print(cp_row("Key Lines for Tourism:"))
print(cp_row("🌊 Cascais Line    - Beach access (Cais do Sodré → Cascais)"))
print(cp_row("🏰 Sintra Line     - UNESCO heritage (Rossio → Sintra)"))
print(cp_row("🌾 Azambuja Line   - North suburbs & Oriente station"))
print(cp_row("🌉 Sado Line       - South-bank CP service (Barreiro → Setúbal)"))
print(cp_row("Excluded: Fertagus is a separate operator, not CP."))
print("├─────────────────────────────────────────────────────────────────────┤")
print(cp_row("Key Observations:"))
print(cp_row("• Real-time GPS tracking of all active trains"))
print(cp_row("• Delay information in seconds (positive = late)"))
print(cp_row("• Official CP GTFS is used for static schedules"))
print(cp_row("• comboios.live supplies community live status data"))
print(cp_row("• LISBOA CP scope is AML suburban CP rail, not Fertagus or long-distance rail"))
print("└─────────────────────────────────────────────────────────────────────┘")


🚆 CP Comboios de Portugal API - Train Network

1. API Connectivity Test
--------------------------------------------------


✅ Stations API responding (Status: 200, Time: 0.17s)

✅ Vehicles API responding (Status: 200, Time: 0.18s)

2. Train Stations (All Portugal)
--------------------------------------------------


✅ Total CP stations: 456
   Available columns: ['code', 'designation', 'latitude', 'longitude', 'region', 'railways']

3. Train Stations (Lisbon Metropolitan Area)
--------------------------------------------------


✅ Stations in AML: 81
ℹ️ AML Bounding Box: Lat [38.5, 39.1], Lon [-9.5, -8.8]


...


4. Real-Time Trains (All Portugal)
--------------------------------------------------


✅ Total active trains: 82
   Available columns: ['trainNumber', 'runDate', 'delay', 'lastStation', 'latitude', 'longitude', 'status', 'hasDisruptions', 'lastStationPlatform', 'gtfs', 'skippedStops', 'timestamp', 'service', 'origin', 'destination']

   📊 Delay Statistics:
      Trains with delays: 70 (85.4%)
      Average delay: 397 seconds
      Max delay: 2676 seconds

5. Real-Time Trains (Lisbon Metropolitan Area)
--------------------------------------------------


✅ Active trains in AML: 29


,trainNumber,status_emoji,status,delay_formatted,latitude,longitude
8,543,🚆,IN_TRANSIT,2m late ⚠️,39.010118,-8.950495
11,720,🚆,IN_TRANSIT,8m late ⚠️,38.965563,-8.982082
13,790,❓,AT_ORIGIN,10m late ⚠️,38.767757,-9.099094
22,4417,🚆,IN_TRANSIT,5m late ⚠️,39.171135,-8.741832
23,4420,🚆,IN_TRANSIT,19s late ⚠️,38.985193,-8.968027
56,16021,🚆,IN_TRANSIT,18s late ⚠️,38.853512,-9.072248
57,16022,❓,AT_STATION,On time ✅,38.778225,-9.099672
58,16428,❓,AT_STATION,3m late ⚠️,38.924034,-9.011380
59,16430,🚆,IN_TRANSIT,2m late ⚠️,38.737723,-9.116415
60,16528,❓,AT_STATION,14s late ⚠️,38.746887,-9.102900



6. CP Lines Serving Lisbon
--------------------------------------------------

📋 Train Lines in the Lisbon Metropolitan Area:

   🌊 Linha de Cascais
      Cais do Sodré ↔ Cascais (coastal route)
      Frequency: ~20 min

   🏰 Linha de Sintra
      Rossio/Oriente ↔ Sintra
      Frequency: ~20 min

   🌾 Linha de Azambuja
      Santa Apolónia/Oriente ↔ Azambuja
      Frequency: ~30 min

   🌉 Linha do Sado
      Barreiro ↔ Setúbal (south-bank CP suburban line)
      Frequency: Variable


📋 CP Comboios de Portugal API Summary
--------------------------------------------------
┌─────────────────────────────────────────────────────────────────────┐
│ 🚆 CP Train Network Statistics                                      │
├─────────────────────────────────────────────────────────────────────┤
│ Total Stations (Portugal):      456                                │
│ Stations in AML:                 81                                │
│ Active Trains (Portugal):        82                           

<div style="background: linear-gradient(to right, #F58228, #FFCD41);
            padding: 3px; color: white; border-radius: 900px; text-align: center;">
</div>

<br>

## **🏁 Runtime Tool Audit (2026-05-18)**

This final section verifies the current runtime registry exported by `tools.__init__.py`. It is intentionally driven by the repository export list, so it fails visibly if a new tool is added without a representative audit payload.

### **Coverage in this notebook**

| Runtime domain | Exported tools covered |
|----------------|------------------------|
| Weather / IPMA | 4 |
| Metro de Lisboa | 6 |
| Carris Metropolitana | 8 |
| CP suburban rail | 6 |
| Multimodal routing | 2 |
| Lisboa Aberta | 5 |
| VisitLisboa | 5 |
| Carris Urban | 8 |
| Web knowledge fallback | 1 |
| **Total** | **45** |

- **Included here:** all **45 exported LangChain tools** across weather, transport, Lisboa Aberta, VisitLisboa, local knowledge search, and constrained web knowledge.
- **Detailed exploration above:** weather and mobility APIs/feeds, because those are the live integrations with the highest operational volatility.
- **Expected variability:** outputs can change with live feeds, service timetables, local GTFS snapshots, vector-store state, web-search availability, and optional Metro credentials.
- **Runtime interpretation:** no-service-at-this-hour, unavailable live feeds, and credential-gated Metro realtime responses are marked as `LIMITED`, not as hard failures.

### **Audit design**

- The cell derives the current export registry directly from `tools.__all__`.
- Each exported tool receives one representative payload matching the current signature.
- Results are summarised as `PASS`, `LIMITED`, `FAIL`, or `ERROR`.

In [13]:
# ===========================================================================
# Runtime Tool Audit - Full Export Registry (2026-05-18)
# ===========================================================================
import json
import time

import tools as lisboa_tools
from tools import __all__ as EXPORTED_TOOL_NAMES

print_header("Runtime Tool Audit - Full Export Registry", "🧪")
print_info(f"tools.__init__ currently exports {len(EXPORTED_TOOL_NAMES)} LangChain tools.")

TOOL_DOMAIN_MAP = {
    "get_weather_warnings": "Weather",
    "get_weather_forecast": "Weather",
    "get_current_weather_summary": "Weather",
    "get_portugal_weather_overview": "Weather",
    "get_metro_status": "Metro",
    "get_metro_wait_time": "Metro",
    "get_metro_line_wait_times": "Metro",
    "find_nearest_metro": "Metro",
    "get_metro_frequency": "Metro",
    "get_all_metro_stations": "Metro",
    "get_real_time_bus_positions": "Carris Metropolitana",
    "get_carris_metropolitana_alerts": "Carris Metropolitana",
    "get_carris_metropolitana_stop_info": "Carris Metropolitana",
    "search_carris_metropolitana_lines": "Carris Metropolitana",
    "find_bus_routes": "Carris Metropolitana",
    "get_bus_realtime_locations": "Carris Metropolitana",
    "get_bus_next_departures": "Carris Metropolitana",
    "find_direct_bus_lines": "Carris Metropolitana",
    "get_train_status": "CP",
    "search_cp_stations": "CP",
    "get_train_schedule": "CP",
    "get_cp_routes": "CP",
    "plan_train_trip": "CP",
    "get_train_frequency": "CP",
    "get_transport_summary": "Multimodal",
    "get_route_between_stations": "Multimodal",
    "find_nearby_services": "Lisboa Aberta",
    "list_available_datasets": "Lisboa Aberta",
    "get_dataset_details": "Lisboa Aberta",
    "find_place_in_datasets": "Lisboa Aberta",
    "list_service_categories": "Lisboa Aberta",
    "search_cultural_events": "VisitLisboa",
    "search_places_attractions": "VisitLisboa",
    "get_event_categories": "VisitLisboa",
    "get_place_categories": "VisitLisboa",
    "search_lisbon_knowledge": "Knowledge/RAG",
    "carris_get_stops": "Carris Urban",
    "carris_get_routes": "Carris Urban",
    "carris_get_next_departures": "Carris Urban",
    "carris_find_routes_between": "Carris Urban",
    "carris_get_realtime_vehicles": "Carris Urban",
    "carris_get_arrivals": "Carris Urban",
    "carris_vehicle_eta": "Carris Urban",
    "carris_get_service_frequency": "Carris Urban",
    "search_history_culture": "Web Knowledge",
}

SAMPLE_PAYLOADS = {
    "get_weather_warnings": {"area": "LSB"},
    "get_weather_forecast": {"days": 3},
    "get_current_weather_summary": {},
    "get_portugal_weather_overview": {"day": 0},
    "get_metro_status": {},
    "get_metro_wait_time": {"station": "Campo Grande"},
    "get_metro_line_wait_times": {"line": "Verde"},
    "find_nearest_metro": {"near_location_name": "Rossio"},
    "get_metro_frequency": {"line": "Verde", "day_type": "weekday", "language": "en"},
    "get_all_metro_stations": {},
    "get_real_time_bus_positions": {},
    "get_carris_metropolitana_alerts": {"language": "en"},
    "get_carris_metropolitana_stop_info": {"stop_id": "010001"},
    "search_carris_metropolitana_lines": {"query": "Oriente"},
    "find_bus_routes": {"origin": "Montijo", "destination": "Oriente"},
    "get_bus_realtime_locations": {},
    "get_bus_next_departures": {"line_id": "1001"},
    "find_direct_bus_lines": {"origin": "Montijo", "destination": "Oriente"},
    "get_train_status": {},
    "search_cp_stations": {"query": "Oriente"},
    "get_train_schedule": {"station_name": "Oriente", "limit": 5},
    "get_cp_routes": {},
    "plan_train_trip": {"origin": "Rossio", "destination": "Sintra"},
    "get_train_frequency": {"route_name": "Sintra"},
    "get_transport_summary": {"language": "en"},
    "get_route_between_stations": {"origin": "Aeroporto", "destination": "Cais do Sodré"},
    "find_nearby_services": {"service_type": "pharmacy", "near_location_name": "Rossio", "max_results": 2, "language": "en"},
    "list_available_datasets": {"category": "transport"},
    "get_dataset_details": {"dataset_name": "Hortas Urbanas"},
    "find_place_in_datasets": {"query": "Biblioteca Camões", "max_results": 2},
    "list_service_categories": {"language": "en"},
    "search_cultural_events": {"query": "music", "max_results": 2, "language": "en"},
    "search_places_attractions": {"query": "Belém", "max_results": 2, "language": "en"},
    "get_event_categories": {"language": "en"},
    "get_place_categories": {"language": "en"},
    "search_lisbon_knowledge": {"query": "Lisboa Card Belém", "max_results": 2},
    "carris_get_stops": {"query": "Belém", "limit": 5},
    "carris_get_routes": {"route_type": "tram", "limit": 5},
    "carris_get_next_departures": {"stop_id": "4109", "limit": 6},
    "carris_find_routes_between": {"origin": "Belém", "destination": "Praça do Comércio"},
    "carris_get_realtime_vehicles": {"route_id": "28E"},
    "carris_get_arrivals": {"stop_id": "4109", "limit": 5},
    "carris_vehicle_eta": {"route_short_name": "15E", "stop_name": "Belém"},
    "carris_get_service_frequency": {"route_short_name": "28E"},
    "search_history_culture": {"query": "Belém Tower Lisbon history", "language": "en"},
}

GLOBAL_LIMITATION_MARKERS = [
    "temporarily unavailable",
    "temporarily unreachable",
    "not configured",
    "credential",
    "credentials",
    "no trains currently scheduled",
    "no trains are currently scheduled",
    "no more departures",
    "no active buses found",
    "no active vehicles",
    "line not operating today",
    "gtfs database unavailable",
    "live feed is temporarily unavailable",
    "using stale cached vehicle data",
    "no real-time arrivals",
]

registry_df = pd.DataFrame(
    [
        {"domain": TOOL_DOMAIN_MAP.get(tool_name, "Unmapped"), "tool": tool_name, "has_payload": tool_name in SAMPLE_PAYLOADS}
        for tool_name in EXPORTED_TOOL_NAMES
    ]
)

print_subheader("Export registry coverage")
display(registry_df.groupby("domain").size().rename("exported_tools").to_frame())

missing_payloads = sorted(set(EXPORTED_TOOL_NAMES) - set(SAMPLE_PAYLOADS))
unmapped_tools = sorted(name for name in EXPORTED_TOOL_NAMES if TOOL_DOMAIN_MAP.get(name) is None)
if missing_payloads:
    print_warning(f"Missing audit payloads: {missing_payloads}")
if unmapped_tools:
    print_warning(f"Unmapped tools: {unmapped_tools}")


def classify_tool_output(result_text: str) -> str:
    """Classify tool output while separating live-data limitations from hard failures."""
    text = str(result_text or "").strip()
    normalized = text.lower()
    if any(marker in normalized for marker in GLOBAL_LIMITATION_MARKERS):
        return "LIMITED"
    if text.startswith(("❌", "âŒ", "Error", "Erro")):
        return "FAIL"
    return "PASS"


def preview_tool_output(result_text: str, max_lines: int = 3, max_chars: int = 240) -> str:
    """Return a compact, table-safe preview of a tool result."""
    preview = " | ".join(str(result_text).splitlines()[:max_lines]).strip()
    return preview[:max_chars] + ("..." if len(preview) > max_chars else "")


tool_audit_results = []
for tool_name in EXPORTED_TOOL_NAMES:
    domain = TOOL_DOMAIN_MAP.get(tool_name, "Unmapped")
    payload = SAMPLE_PAYLOADS.get(tool_name)
    if payload is None:
        tool_audit_results.append(
            {
                "domain": domain,
                "tool": tool_name,
                "status": "ERROR",
                "duration_s": 0.0,
                "payload": "{}",
                "preview": "No representative payload configured for this exported tool.",
            }
        )
        continue

    start_time = time.time()
    try:
        tool = getattr(lisboa_tools, tool_name)
        result_text = tool.invoke(payload)
        status = classify_tool_output(result_text)
        preview = preview_tool_output(result_text)
    except Exception as exc:
        status = "ERROR"
        preview = preview_tool_output(f"{type(exc).__name__}: {exc}")

    tool_audit_results.append(
        {
            "domain": domain,
            "tool": tool_name,
            "status": status,
            "duration_s": round(time.time() - start_time, 2),
            "payload": json.dumps(payload, ensure_ascii=False),
            "preview": preview,
        }
    )

tool_audit_df = pd.DataFrame(tool_audit_results)

print_subheader("Tool results")
display(tool_audit_df)

tool_audit_summary = tool_audit_df.groupby(["domain", "status"]).size().unstack(fill_value=0).sort_index()
display(tool_audit_summary)

pass_count = int((tool_audit_df["status"] == "PASS").sum())
limited_count = int((tool_audit_df["status"] == "LIMITED").sum())
fail_count = int((tool_audit_df["status"] == "FAIL").sum())
error_count = int((tool_audit_df["status"] == "ERROR").sum())
expected_count = len(EXPORTED_TOOL_NAMES)
executed_count = len(tool_audit_df)

print_subheader("Audit summary")
print_success(f"Audited {executed_count}/{expected_count} exported tools from tools.__init__.py.")
print(f"   PASS:    {pass_count}")
print(f"   LIMITED: {limited_count}")
print(f"   FAIL:    {fail_count}")
print(f"   ERROR:   {error_count}")

if fail_count or error_count:
    print_warning("Review any non-green rows above before treating the notebook as fully updated.")
else:
    print_success("No hard failures detected in the full runtime tool audit.")

print_info("LIMITED rows represent live-service, credential, no-service-window, or cached-feed constraints, not code regressions.")



🧪 Runtime Tool Audit - Full Export Registry
ℹ️ tools.__init__ currently exports 45 LangChain tools.

Export registry coverage
--------------------------------------------------


,exported_tools
domain,
CP,6
Carris Metropolitana,8
Carris Urban,8
Knowledge/RAG,1
Lisboa Aberta,5
Metro,6
Multimodal,2
VisitLisboa,4
Weather,4


📥 Initializing KnowledgeBase...


   Importing AI libraries (this may take a moment)...


   Loading Embeddings: BAAI/bge-m3...


...

   DB Path: C:\Users\andre\OneDrive - NOVAIMS\[MDSAA-DS]_Thesis\LISBOA_MultiAgentSystem\data\vector_db



Tool results
--------------------------------------------------


,domain,tool,status,duration_s,payload,preview
0,Weather,get_weather_warnings,PASS,0.05,"{""area"": ""LSB""}",✅ No active weather warnings for Lisbon.
1,Weather,get_weather_forecast,PASS,0.00,"{""days"": 3}",🌤️ Weather Forecast for Lisbon | ======================================== | 📅 Updated: 2026-05-1...
2,Weather,get_current_weather_summary,PASS,0.00,{},🌤️ Lisbon Weather Summary | ======================================== |
3,Weather,get_portugal_weather_overview,PASS,0.01,"{""day"": 0}",🇵🇹 Portugal Weather Overview - Today | 📅 Forecast date — 2026-05-18 | 🔄 Updated: 2026-05-18T12:3...
4,Metro,get_metro_status,PASS,0.17,{},🚇 Metro de Lisboa Status (Official API) | ============================================= |
5,Metro,get_metro_wait_time,PASS,0.18,"{""station"": ""Campo Grande""}",🚇 Metro Wait Times at Campo Grande | ================================================== |
6,Metro,get_metro_line_wait_times,PASS,0.17,"{""line"": ""Verde""}",🟢 Green Line (Telheiras ↔ Cais do Sodré) - Wait Times | ========================================...
7,Metro,find_nearest_metro,PASS,0.00,"{""near_location_name"": ""Rossio""}",### 🚇 **Nearest Metro Stations** | | - 🟢 **Rossio**
8,Metro,get_metro_frequency,PASS,0.18,"{""line"": ""Verde"", ""day_type"": ""weekday"", ""language"": ""en""}",🟢 **Green Line (Telheiras ↔ Cais do Sodré)** | | ✅ **Direct answer:** the scheduled frequency f...
9,Metro,get_all_metro_stations,PASS,0.00,{},### 🚇 **Lisbon Metro** | | **🟡 Yellow Line — Rato ↔ Odivelas**


status,LIMITED,PASS
domain,,
CP,0,6
Carris Metropolitana,1,7
Carris Urban,0,8
Knowledge/RAG,0,1
Lisboa Aberta,0,5
Metro,0,6
Multimodal,0,2
VisitLisboa,0,4
Weather,0,4



Audit summary
--------------------------------------------------
✅ Audited 45/45 exported tools from tools.__init__.py.
   PASS:    44
   LIMITED: 1
   FAIL:    0
   ERROR:   0
✅ No hard failures detected in the full runtime tool audit.
ℹ️ LIMITED rows represent live-service, credential, no-service-window, or cached-feed constraints, not code regressions.


---

## **Final notes**

- As of **2026-05-18**, this notebook covers raw API/feed exploration for the weather and mobility stack plus a full audit of the **45 tools** exported by `tools.__init__.py`.
- Current-source checks used for the Markdown were IPMA open data, the Metro API Store, Carris Metropolitana API documentation/current endpoints, the CP GTFS feed, the Carris GTFS/GTFS-RT endpoints, and the local JSON/SQLite snapshots in this repository.
- The final audit is registry-driven: if the exported tool list changes, the coverage table exposes missing payloads or unmapped tools.
- Live outputs vary with service schedules, feed freshness, local GTFS snapshots, vector-store state, web-search availability, and optional Metro credentials.
- Re-run the final runtime audit cell after meaningful changes to `tools.__init__.py`, API modules, transport modules, VisitLisboa/Lisboa Aberta tools, or the web-knowledge tool.